# AED v10 — atenção temporal, fala normal e holdout bloqueado

O AED v10 parte do v9, que atingiu aproximadamente:

- acurácia: 86,98%;
- F1 macro: 86,47%;
- menor F1: 79,53%.

Esta versão tenta atacar os erros restantes sem usar o conjunto final
para escolher arquitetura, pesos de fusão ou limiares.

Mudanças principais:

1. quatro divisões por grupo:
   - treino;
   - validação;
   - calibração;
   - holdout final;
2. TUT Rare e AOB desenvolvimento entram somente no treino;
3. Speech Commands adiciona milhares de falas normais como negativos;
4. a sequência YAMNet usa pooling por atenção aprendido;
5. especialista adicional `grito × ambiente_normal`;
6. augmentation leve em tempo de treino, sem duplicar arquivos;
7. seleção da etapa somente pela validação;
8. pesos de fusão e limiares somente pela calibração;
9. holdout final avaliado apenas depois de todas as decisões;
10. avaliação externa AOB;
11. exportação TFLite Float32 e Float16 com paridade.

Critérios de aceitação:

- acurácia no holdout `>= 90%`;
- F1 macro no holdout `>= 90%`;
- menor F1 de classe `>= 80%`.

Os 90% continuam sendo uma meta, não um resultado forçado.

In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg git
!pip -q install         kagglehub         librosa==0.11.0         soundfile         scikit-learn         pandas         matplotlib         tqdm         requests         tensorflow-hub         tensorflow-datasets         datasets         huggingface_hub

In [ ]:
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any

import gc
import io
import json
import math
import random
import re
import shutil
import subprocess
import time
import zipfile

import kagglehub
import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import soundfile as sf
import tensorflow as tf
import tensorflow_datasets as tfds
import tensorflow_hub as hub

from datasets import Features, Value, load_dataset
from google.colab import files
from huggingface_hub import hf_hub_download
from IPython.display import Audio, display
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

if not tf.config.list_physical_devices("GPU"):
    raise RuntimeError(
        "Ative uma GPU T4 no Colab."
    )

for gpu in tf.config.list_physical_devices("GPU"):
    try:
        tf.config.experimental.set_memory_growth(
            gpu,
            True,
        )
    except RuntimeError:
        pass

tf.keras.mixed_precision.set_global_policy(
    "float32"
)

print(
    "Política numérica:",
    tf.keras.mixed_precision.global_policy(),
)

## 1. Configuração

In [ ]:
@dataclass(frozen=True)
class Configuracao:
    sr: int = 16000
    duracao: float = 3.0
    passo: float = 0.50

    n_mfcc: int = 40
    n_fft: int = 512
    hop_length: int = 256
    patches_yamnet: int = 6

    max_arquivos_por_classe: int = 5000
    max_janelas_por_arquivo: int = 3
    max_janelas_suplementares: int = 2

    max_fsd_por_classe: int = 1800
    max_fsd_impacto: int = 1600
    max_fsd_normal: int = 3000

    max_baby_kaggle: int = 1600
    max_donate_cry: int = 900
    max_speech_commands: int = 6500

    lote: int = 48
    epocas_etapa_1: int = 70
    epocas_etapa_2: int = 32
    paciencia: int = 10
    seed: int = 42

    tentativas_split: int = 450

    proporcao_treino: float = 0.65
    proporcao_validacao: float = 0.15
    proporcao_calibracao: float = 0.10
    proporcao_holdout: float = 0.10

    max_hard_normal: int = 2000
    max_hard_grito: int = 1200
    max_hard_choro: int = 1200
    max_hard_impacto: int = 1400
    repeticoes_dificeis: int = 3

    ruido_yamnet: float = 0.008
    ruido_acustico: float = 0.012
    prob_mascarar_patch: float = 0.20
    prob_dropout_acustico: float = 0.15

    meta_accuracy: float = 0.90
    meta_f1_macro: float = 0.90
    meta_f1_classe: float = 0.80
    meta_paridade: float = 0.99

    usar_aob: bool = True
    usar_tau_privacy: bool = True
    usar_tut: bool = True
    usar_reforco_manual: bool = False
    usar_fsd50k: bool = True
    usar_baby_kaggle: bool = True
    usar_donate_cry: bool = True
    usar_speech_commands: bool = True


CFG = Configuracao()

CLASSES = [
    "ambiente_normal",
    "grito_panico",
    "choro_distress",
    "impacto",
    "vidro_quebrando",
    "tiro",
]

ID = {
    classe: indice
    for indice, classe in enumerate(CLASSES)
}

FAMILIA = {
    "ambiente_normal": 0,
    "grito_panico": 1,
    "choro_distress": 1,
    "impacto": 2,
    "vidro_quebrando": 2,
    "tiro": 2,
}

VOCAL = {
    "grito_panico": 0,
    "choro_distress": 1,
}

TRANSITORIO = {
    "impacto": 0,
    "vidro_quebrando": 1,
    "tiro": 2,
}

RARAS = set(TRANSITORIO)

PESOS_AMOSTRAGEM = {
    "ambiente_normal": 0.20,
    "grito_panico": 0.16,
    "choro_distress": 0.17,
    "impacto": 0.19,
    "vidro_quebrando": 0.15,
    "tiro": 0.13,
}

if not np.isclose(
    sum(PESOS_AMOSTRAGEM.values()),
    1.0,
):
    raise RuntimeError(
        "Pesos de amostragem inválidos."
    )

SOMA_SPLITS = (
    CFG.proporcao_treino
    + CFG.proporcao_validacao
    + CFG.proporcao_calibracao
    + CFG.proporcao_holdout
)

if not np.isclose(SOMA_SPLITS, 1.0):
    raise RuntimeError(
        "As proporções dos splits devem somar 1."
    )

RAIZ = Path("/content/aed_v10")
DADOS = RAIZ / "datasets"
CACHE = RAIZ / "cache"
MODELOS = RAIZ / "modelos"
RESULTADOS = RAIZ / "resultados"
ENTREGA = RAIZ / "entrega"

ESC_DIR = DADOS / "esc50"
TUT_DIR = DADOS / "tut_rare"
AOB_DEV = DADOS / "aob_dev"
AOB_EVAL = DADOS / "aob_eval"
TAU_DIR = DADOS / "tau_privacy"
REFORCO = DADOS / "reforco_manual"
DONATE_DIR = DADOS / "donate_a_cry"
SPEECH_COMMANDS_DIR = DADOS / "speech_commands"

for pasta in (
    RAIZ,
    DADOS,
    CACHE,
    MODELOS,
    RESULTADOS,
    ENTREGA,
):
    pasta.mkdir(
        parents=True,
        exist_ok=True,
    )

random.seed(CFG.seed)
np.random.seed(CFG.seed)
tf.random.set_seed(CFG.seed)

display(
    pd.DataFrame([asdict(CFG)])
)

## 2. Download dos datasets

In [ ]:
ASVP = Path(
    kagglehub.dataset_download(
        (
            "dejolilandry/"
            "asvpesdspeech-nonspeech-emotional-utterances"
        )
    )
)

GUNSHOT = Path(
    kagglehub.dataset_download(
        "emrahaydemr/gunshot-audio-dataset"
    )
)

URBAN = Path(
    kagglehub.dataset_download(
        "chrisfilo/urbansound8k"
    )
)

SCREAM = Path(
    kagglehub.dataset_download(
        (
            "aananehsansiam/"
            "audio-dataset-of-scream-and-non-scream"
        )
    )
)

print("ASVP:", ASVP)
print("Gunshot:", GUNSHOT)
print("UrbanSound8K:", URBAN)
print("Scream:", SCREAM)

In [ ]:
import time


def zip_valido(
    caminho: Path,
) -> bool:
    return (
        caminho.exists()
        and caminho.stat().st_size > 0
        and zipfile.is_zipfile(caminho)
    )


def baixar_zip(
    urls: str | list[str],
    destino: Path,
    obrigatorio: bool = True,
    tentativas_por_url: int = 4,
) -> bool:
    if zip_valido(destino):
        print("Já disponível:", destino)
        return True

    if isinstance(urls, str):
        urls = [urls]

    destino.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporario = destino.with_suffix(
        destino.suffix + ".part"
    )

    # Pode ter terminado o download, mas não ter sido renomeado.
    if zip_valido(temporario):
        temporario.replace(destino)
        print("Download parcial já estava completo:", destino)
        return True

    ultimo_erro = None

    for url in urls:
        for tentativa in range(
            1,
            tentativas_por_url + 1,
        ):
            try:
                tamanho_existente = (
                    temporario.stat().st_size
                    if temporario.exists()
                    else 0
                )

                cabecalhos = {
                    "User-Agent": (
                        "Mozilla/5.0 "
                        "violence-edge-ai-aed-v8"
                    )
                }

                if tamanho_existente > 0:
                    cabecalhos["Range"] = (
                        f"bytes={tamanho_existente}-"
                    )

                print(
                    "\nBaixando:",
                    destino.name,
                )
                print(
                    "Rota:",
                    url,
                )
                print(
                    f"Tentativa: {tentativa}/"
                    f"{tentativas_por_url}",
                )
                print(
                    "Já baixado:",
                    f"{tamanho_existente / 1024**2:.1f} MB",
                )

                with requests.get(
                    url,
                    stream=True,
                    timeout=(45, 600),
                    verify=True,
                    allow_redirects=True,
                    headers=cabecalhos,
                ) as resposta:
                    # O servidor pode informar que o arquivo já terminou.
                    if (
                        resposta.status_code == 416
                        and zip_valido(temporario)
                    ):
                        temporario.replace(destino)
                        return True

                    resposta.raise_for_status()

                    # 206 significa que o servidor aceitou continuar.
                    continuar = (
                        tamanho_existente > 0
                        and resposta.status_code == 206
                    )

                    if continuar:
                        modo = "ab"
                        inicio_barra = tamanho_existente
                    else:
                        modo = "wb"
                        inicio_barra = 0

                    restante = int(
                        resposta.headers.get(
                            "content-length",
                            0,
                        )
                    )

                    total = (
                        inicio_barra + restante
                        if restante > 0
                        else None
                    )

                    with temporario.open(
                        modo
                    ) as arquivo:
                        with tqdm(
                            total=total,
                            initial=inicio_barra,
                            unit="B",
                            unit_scale=True,
                            desc=destino.name,
                        ) as barra:
                            for bloco in resposta.iter_content(
                                chunk_size=4 * 1024 * 1024
                            ):
                                if not bloco:
                                    continue

                                arquivo.write(bloco)
                                barra.update(len(bloco))

                if not zip_valido(temporario):
                    raise RuntimeError(
                        "O arquivo recebido não é um ZIP válido."
                    )

                temporario.replace(destino)

                print(
                    "Download concluído:",
                    destino,
                )

                return True

            except Exception as erro:
                ultimo_erro = erro

                print(
                    "Falhou:",
                    type(erro).__name__,
                    erro,
                )

                # Mantém o .part para tentar continuar depois.
                espera = min(
                    60,
                    5 * (2 ** (tentativa - 1)),
                )

                print(
                    f"Nova tentativa em {espera}s..."
                )

                time.sleep(espera)

    mensagem = (
        f"Não foi possível baixar {destino.name}. "
        f"Último erro: {ultimo_erro}"
    )

    if obrigatorio:
        raise RuntimeError(mensagem)

    print("\nAVISO:", mensagem)
    print(
        "Este dataset será ignorado e "
        "o notebook continuará."
    )

    return False


def extrair_zip(
    arquivo_zip: Path,
    destino: Path,
) -> bool:
    if not zip_valido(arquivo_zip):
        print(
            "Extração ignorada; ZIP ausente:",
            arquivo_zip,
        )
        return False

    marcador = destino / ".extraido"

    if marcador.exists():
        print(
            "Já extraído:",
            destino,
        )
        return True

    destino.mkdir(
        parents=True,
        exist_ok=True,
    )

    print(
        "Extraindo:",
        arquivo_zip.name,
    )

    with zipfile.ZipFile(
        arquivo_zip
    ) as arquivo:
        arquivo.extractall(destino)

    marcador.touch()

    print(
        "Extração concluída:",
        destino,
    )

    return True


# ============================================================
# ESC-50 — obrigatório
# ============================================================

ZIP_ESC = DADOS / "esc50.zip"

esc_disponivel = baixar_zip(
    [
        (
            "https://github.com/karolpiczak/"
            "ESC-50/archive/refs/heads/master.zip"
        ),
    ],
    ZIP_ESC,
    obrigatorio=True,
)

if esc_disponivel:
    extrair_zip(
        ZIP_ESC,
        ESC_DIR,
    )


# ============================================================
# TUT Rare — opcional
# O notebook continua caso o Zenodo permaneça indisponível.
# ============================================================

TUT_DISPONIVEL = False

if CFG.usar_tut:
    ZIP_TUT = DADOS / "tut_rare.zip"

    nome_tut = (
        "TUT-rare-sound-events-2017-development."
        "source_data_events.zip"
    )

    TUT_DISPONIVEL = baixar_zip(
        [
            (
                "https://zenodo.org/api/records/401395/"
                f"files/{nome_tut}/content"
            ),
            (
                "https://zenodo.org/records/401395/files/"
                f"{nome_tut}?download=1"
            ),
        ],
        ZIP_TUT,
        obrigatorio=False,
        tentativas_por_url=4,
    )

    if TUT_DISPONIVEL:
        extrair_zip(
            ZIP_TUT,
            TUT_DIR,
        )

print(
    "TUT disponível:",
    TUT_DISPONIVEL,
)


# ============================================================
# Dataset-AOB — obrigatório
# ============================================================

if CFG.usar_aob:
    ZIP_AOB_DEV = (
        DADOS / "aob_dev.zip"
    )

    ZIP_AOB_EVAL = (
        DADOS / "aob_eval.zip"
    )

    aob_dev_disponivel = baixar_zip(
        [
            (
                "https://zenodo.org/api/records/4319802/"
                "files/DATASET_AOB.zip/content"
            ),
            (
                "https://zenodo.org/records/4319802/files/"
                "DATASET_AOB.zip?download=1"
            ),
        ],
        ZIP_AOB_DEV,
        obrigatorio=True,
    )

    aob_eval_disponivel = baixar_zip(
        [
            (
                "https://zenodo.org/api/records/4319802/"
                "files/EVALUATION_dataset_AOB.zip/content"
            ),
            (
                "https://zenodo.org/records/4319802/files/"
                "EVALUATION_dataset_AOB.zip?download=1"
            ),
        ],
        ZIP_AOB_EVAL,
        obrigatorio=True,
    )

    if aob_dev_disponivel:
        extrair_zip(
            ZIP_AOB_DEV,
            AOB_DEV,
        )

    if aob_eval_disponivel:
        extrair_zip(
            ZIP_AOB_EVAL,
            AOB_EVAL,
        )


# ============================================================
# TAU Privacy — obrigatório
# ============================================================

if CFG.usar_tau_privacy:
    ZIP_TAU = (
        DADOS / "tau_privacy.zip"
    )

    tau_disponivel = baixar_zip(
        [
            (
                "https://zenodo.org/api/records/10378145/"
                "files/final_data_wav_split.zip/content"
            ),
            (
                "https://zenodo.org/records/10378145/files/"
                "final_data_wav_split.zip?download=1"
            ),
        ],
        ZIP_TAU,
        obrigatorio=True,
    )

    if tau_disponivel:
        extrair_zip(
            ZIP_TAU,
            TAU_DIR,
        )


print("\nDownloads concluídos.")
print("ESC-50:", esc_disponivel)
print("TUT Rare:", TUT_DISPONIVEL)

if CFG.usar_aob:
    print("AOB desenvolvimento:", aob_dev_disponivel)
    print("AOB avaliação:", aob_eval_disponivel)

if CFG.usar_tau_privacy:
    print("TAU Privacy:", tau_disponivel)

## 3. Reforço manual opcional

## 4. Metadados unificados

In [ ]:
EXTENSOES = {
    ".wav",
    ".mp3",
    ".ogg",
    ".flac",
    ".m4a",
}


def audios_em(
    pasta: Path | None,
) -> list[Path]:
    if pasta is None or not pasta.exists():
        return []

    return sorted(
        caminho
        for caminho in pasta.rglob("*")
        if (
            caminho.is_file()
            and caminho.suffix.lower()
            in EXTENSOES
        )
    )


def token(texto: str) -> str:
    return re.sub(
        r"[^a-z0-9]+",
        "_",
        str(texto).lower(),
    ).strip("_")


def base_gravacao(nome: str) -> str:
    nome = token(nome)
    nome = re.sub(
        r"_(speech|no_speech|nospeech).*$",
        "",
        nome,
    )
    nome = re.sub(
        r"_(chunk|segment|slice|part)_?\d+$",
        "",
        nome,
    )
    return nome or "gravacao"


def adicionar(
    registros: list[dict[str, Any]],
    arquivo: Path,
    classe: str,
    fonte: str,
    grupo: str,
    split_fixo: str = "",
    extra: str = "",
) -> None:
    arquivo = Path(arquivo)

    if arquivo.exists() and classe in ID:
        registros.append(
            {
                "arquivo": str(arquivo.resolve()),
                "classe": classe,
                "classe_id": ID[classe],
                "familia": FAMILIA[classe],
                "perigo": int(
                    classe != "ambiente_normal"
                ),
                "fonte": fonte,
                "grupo": str(grupo),
                "split_fixo": split_fixo,
                "extra": str(extra),
            }
        )


registros = []


# ASVP
mapa_asvp = {
    "02": "ambiente_normal",
    "04": "choro_distress",
    "06": "grito_panico",
}

for arquivo in audios_em(ASVP):
    numeros = re.findall(
        r"\d+",
        arquivo.stem,
    )

    if len(numeros) < 6:
        continue

    emocao = numeros[2].zfill(2)
    ator = numeros[5].zfill(3)
    classe = mapa_asvp.get(emocao)

    if classe is None:
        continue

    adicionar(
        registros,
        arquivo,
        classe,
        "ASVP",
        f"asvp:ator_{ator}",
        extra=emocao,
    )


# Scream / non-scream
for arquivo in audios_em(SCREAM):
    relativo = arquivo.relative_to(
        SCREAM
    )

    pastas = [
        token(parte)
        for parte in relativo.parts[:-1]
    ]

    if any(
        nome in {
            "non_scream",
            "nonscream",
            "no_scream",
            "not_scream",
        }
        for nome in pastas
    ):
        classe = "ambiente_normal"

    elif any(
        nome in {
            "scream",
            "screams",
        }
        for nome in pastas
    ):
        classe = "grito_panico"

    else:
        continue

    adicionar(
        registros,
        arquivo,
        classe,
        "ScreamDataset",
        (
            f"scream:{classe}:"
            f"{token(arquivo.stem)}"
        ),
        extra="/".join(pastas),
    )


# Gunshot
for arquivo in audios_em(GUNSHOT):
    relativo = arquivo.relative_to(
        GUNSHOT
    )

    origem = token(
        relativo.parts[0]
        if relativo.parts
        else arquivo.parent.name
    )

    sessao = re.sub(
        r"\s*\(\d+\)$",
        "",
        arquivo.stem,
    )
    sessao = token(sessao)

    adicionar(
        registros,
        arquivo,
        "tiro",
        "GunshotKaggle",
        f"gunshot:{origem}:{sessao}",
        extra=origem,
    )


# ESC-50
csvs_esc = list(
    ESC_DIR.rglob("esc50.csv")
)
pastas_esc = [
    pasta
    for pasta in ESC_DIR.rglob("audio")
    if pasta.is_dir()
]

if not csvs_esc or not pastas_esc:
    raise RuntimeError(
        "ESC-50 incompleto."
    )

tabela_esc = pd.read_csv(
    csvs_esc[0]
)
audio_esc = pastas_esc[0]

mapa_esc = {
    "crying_baby": "choro_distress",
    "glass_breaking": "vidro_quebrando",
    "door_wood_knock": "impacto",
}

normais_esc = {
    "clapping",
    "hand_saw",
    "door_wood_creaks",
    "fireworks",
    "chainsaw",
    "siren",
    "engine",
    "vacuum_cleaner",
    "washing_machine",
    "clock_alarm",
    "can_opening",
    "mouse_click",
    "keyboard_typing",
    "footsteps",
    "laughing",
    "breathing",
    "coughing",
    "sneezing",
    "snoring",
    "rain",
    "sea_waves",
    "wind",
    "thunderstorm",
    "airplane",
    "helicopter",
    "train",
}

for linha in tabela_esc.itertuples():
    categoria = str(linha.category)

    if categoria in mapa_esc:
        classe = mapa_esc[categoria]
    elif categoria in normais_esc:
        classe = "ambiente_normal"
    else:
        continue

    adicionar(
        registros,
        audio_esc / str(linha.filename),
        classe,
        "ESC-50",
        f"esc50:{linha.src_file}",
        extra=categoria,
    )


# UrbanSound8K
csvs_urban = list(
    URBAN.rglob("UrbanSound8K.csv")
)

if csvs_urban:
    tabela_urban = pd.read_csv(
        csvs_urban[0]
    )
    raiz_audio = (
        csvs_urban[0].parent.parent
        / "audio"
    )

    for _, linha in tabela_urban.iterrows():
        nome = str(linha["class"])

        if nome == "gun_shot":
            classe = "tiro"
        elif nome in {
            "jackhammer",
            "drilling",
            "siren",
            "car_horn",
            "dog_bark",
            "street_music",
            "children_playing",
            "air_conditioner",
            "engine_idling",
        }:
            classe = "ambiente_normal"
        else:
            continue

        arquivo = (
            raiz_audio
            / f"fold{int(linha['fold'])}"
            / str(linha["slice_file_name"])
        )

        fsid = str(
            linha.get(
                "fsID",
                str(
                    linha["slice_file_name"]
                ).split("-")[0],
            )
        )

        adicionar(
            registros,
            arquivo,
            classe,
            "UrbanSound8K",
            f"urban:fsid_{fsid}",
            extra=nome,
        )


# TUT Rare
if CFG.usar_tut:
    for arquivo in audios_em(TUT_DIR):
        texto = token(str(arquivo))

        if any(
            termo in texto
            for termo in (
                "babycry",
                "baby_cry",
                "babycrying",
            )
        ):
            classe = "choro_distress"
        elif any(
            termo in texto
            for termo in (
                "glassbreak",
                "glass_break",
                "glassbreaking",
            )
        ):
            classe = "vidro_quebrando"
        elif any(
            termo in texto
            for termo in (
                "gunshot",
                "gun_shot",
                "gunfire",
            )
        ):
            classe = "tiro"
        else:
            continue

        adicionar(
            registros,
            arquivo,
            classe,
            "TUT_Rare",
            f"tut:{classe}:{arquivo.stem}",
            split_fixo="treino",
            extra=classe,
        )


# Dataset-AOB
def classe_aob(
    caminho: Path,
) -> str | None:
    texto = token(str(caminho))

    if any(
        termo in texto
        for termo in (
            "glass_breaking",
            "glassbreaking",
            "glass_break",
        )
    ):
        return "vidro_quebrando"

    if any(
        termo in texto
        for termo in (
            "gun_shot",
            "gunshot",
        )
    ):
        return "tiro"

    if any(
        termo in texto
        for termo in (
            "screams",
            "scream",
        )
    ):
        return "grito_panico"

    if any(
        termo in texto
        for termo in (
            "alarm_siren",
            "children_playing",
            "dog_bark",
            "engine",
            "footsteps",
            "metro_train",
            "rain",
        )
    ):
        return "ambiente_normal"

    return None


if CFG.usar_aob:
    for raiz_aob, split_fixo in (
        (AOB_DEV, "treino"),
        (AOB_EVAL, "externo"),
    ):
        for arquivo in audios_em(
            raiz_aob
        ):
            classe = classe_aob(
                arquivo
            )

            if classe is None:
                continue

            relativo = arquivo.relative_to(
                raiz_aob
            )

            categoria = token(
                str(relativo.parent)
            )
            grupo_base = base_gravacao(
                relativo.stem
            )

            adicionar(
                registros,
                arquivo,
                classe,
                "Dataset-AOB",
                (
                    f"aob:{split_fixo or 'dev'}:"
                    f"{categoria}:{grupo_base}"
                ),
                split_fixo=split_fixo,
                extra=str(relativo.parent),
            )


# TAU Privacy
def classe_tau(
    caminho: Path,
) -> str | None:
    texto = token(str(caminho))

    if "glass_breaking" in texto:
        return "vidro_quebrando"

    if (
        "gun_shot" in texto
        or "gunshot" in texto
    ):
        return "tiro"

    if "slam" in texto:
        return "impacto"

    if "crying" in texto:
        return "choro_distress"

    if any(
        termo in texto
        for termo in (
            "dog_barking",
            "cough",
            "applause",
            "dished_pot_pan",
            "toilet_flush",
            "cat_meowing",
            "doorbell",
            "drill",
        )
    ):
        return "ambiente_normal"

    return None


if CFG.usar_tau_privacy:
    for arquivo in audios_em(TAU_DIR):
        classe = classe_tau(arquivo)

        if classe is None:
            continue

        relativo = arquivo.relative_to(
            TAU_DIR
        )
        evento = token(
            relativo.parent.parent.name
            if len(relativo.parts) >= 3
            else classe
        )
        base = base_gravacao(
            arquivo.stem
        )

        adicionar(
            registros,
            arquivo,
            classe,
            "TAU-Privacy",
            f"tau:{evento}:{base}",
            extra=str(
                arquivo.relative_to(
                    TAU_DIR
                ).parent
            ),
        )


# Reforço manual
if CFG.usar_reforco_manual:
    for classe in CLASSES:
        for pasta in REFORCO.rglob(
            classe
        ):
            if not pasta.is_dir():
                continue

            for arquivo in audios_em(
                pasta
            ):
                adicionar(
                    registros,
                    arquivo,
                    classe,
                    "ReforcoManual",
                    (
                        f"manual:{classe}:"
                        f"{token(arquivo.stem)}"
                    ),
                    extra=classe,
                )


meta = pd.DataFrame(registros)

if meta.empty:
    raise RuntimeError(
        "Nenhum áudio encontrado."
    )

meta = meta.drop_duplicates(
    subset=["arquivo", "classe"]
)

desenvolvimento = meta[
    meta["split_fixo"] != "externo"
].copy()

treino_fixo = desenvolvimento[
    desenvolvimento["split_fixo"] == "treino"
].copy()

desenvolvimento = desenvolvimento[
    desenvolvimento["split_fixo"] != "treino"
].copy()

externo = meta[
    meta["split_fixo"] == "externo"
].copy()

partes = []

for classe in CLASSES:
    parte = desenvolvimento[
        desenvolvimento["classe"]
        == classe
    ].copy()

    if (
        len(parte)
        > CFG.max_arquivos_por_classe
    ):
        parte = parte.sample(
            CFG.max_arquivos_por_classe,
            random_state=(
                CFG.seed + ID[classe]
            ),
        )

    partes.append(parte)

desenvolvimento = pd.concat(
    partes,
    ignore_index=True,
)

display(
    desenvolvimento.groupby(
        ["classe", "fonte"]
    )
    .size()
    .unstack(fill_value=0)
    .reindex(
        CLASSES,
        fill_value=0,
    )
)

resumo = (
    desenvolvimento.groupby(
        "classe"
    )
    .agg(
        arquivos=("arquivo", "count"),
        grupos=("grupo", "nunique"),
    )
    .reindex(
        CLASSES,
        fill_value=0,
    )
)

display(resumo)

faltantes = [
    classe
    for classe in CLASSES
    if resumo.loc[
        classe,
        "arquivos",
    ] == 0
]

if faltantes:
    raise RuntimeError(
        f"Classes sem dados: {faltantes}"
    )

desenvolvimento.to_csv(
    RESULTADOS
    / "fontes_desenvolvimento.csv",
    index=False,
)

treino_fixo.to_csv(
    RESULTADOS
    / "fontes_treino_fixo.csv",
    index=False,
)

externo.to_csv(
    RESULTADOS
    / "fontes_externo_aob.csv",
    index=False,
)

## 5. Auditoria manual

In [ ]:
for classe in CLASSES:
    print("\n" + "=" * 72)
    print("Classe:", classe)

    parte = desenvolvimento[
        desenvolvimento["classe"]
        == classe
    ]

    for linha in (
        parte.sample(
            min(2, len(parte)),
            random_state=(
                CFG.seed + ID[classe]
            ),
        )
        .itertuples()
    ):
        print(
            linha.fonte,
            "|",
            Path(linha.arquivo).name,
            "|",
            linha.extra,
        )
        display(
            Audio(linha.arquivo)
        )

In [ ]:
APROVAR_AUDITORIA = True

if not APROVAR_AUDITORIA:
    raise RuntimeError(
        "Ouça as amostras. Depois altere "
        "APROVAR_AUDITORIA para True."
    )

## 6. Split global por grupo, com calibração e holdout

O holdout final não participa da escolha da etapa, da combinação
hierárquica nem dos limiares.

TUT Rare e AOB desenvolvimento ficam exclusivamente no treino.

In [ ]:
ALVOS_SPLIT = {
    "treino": CFG.proporcao_treino,
    "validacao": CFG.proporcao_validacao,
    "calibracao": CFG.proporcao_calibracao,
    "holdout": CFG.proporcao_holdout,
}


def pontuar_split(
    tabela: pd.DataFrame,
) -> float:
    totais = (
        tabela.groupby("classe")
        .size()
        .reindex(CLASSES, fill_value=0)
    )

    score = 0.0

    for split, alvo in ALVOS_SPLIT.items():
        parte = tabela[
            tabela["split"] == split
        ]

        score += abs(
            len(parte) / len(tabela) - alvo
        )

        contagem = (
            parte.groupby("classe")
            .size()
            .reindex(CLASSES, fill_value=0)
        )

        proporcao = (
            contagem
            / totais.replace(0, np.nan)
        ).fillna(0.0)

        score += float(
            np.abs(
                proporcao - alvo
            ).mean()
        )

    return float(score)


def criar_split_global(
    tabela: pd.DataFrame,
):
    melhor = None
    melhor_score = float("inf")

    proporcao_restante = (
        1.0
        - CFG.proporcao_treino
    )

    fracao_validacao_no_restante = (
        CFG.proporcao_validacao
        / proporcao_restante
    )

    fracao_calibracao_no_final = (
        CFG.proporcao_calibracao
        / (
            CFG.proporcao_calibracao
            + CFG.proporcao_holdout
        )
    )

    for tentativa in range(
        CFG.tentativas_split
    ):
        semente = CFG.seed + tentativa

        divisor_1 = GroupShuffleSplit(
            n_splits=1,
            train_size=CFG.proporcao_treino,
            random_state=semente,
        )

        idx_treino, idx_restante = next(
            divisor_1.split(
                tabela,
                groups=tabela["grupo"],
            )
        )

        restante = tabela.iloc[
            idx_restante
        ]

        divisor_2 = GroupShuffleSplit(
            n_splits=1,
            train_size=fracao_validacao_no_restante,
            random_state=semente + 10000,
        )

        idx_val_rel, idx_final_rel = next(
            divisor_2.split(
                restante,
                groups=restante["grupo"],
            )
        )

        final = restante.iloc[
            idx_final_rel
        ]

        divisor_3 = GroupShuffleSplit(
            n_splits=1,
            train_size=fracao_calibracao_no_final,
            random_state=semente + 20000,
        )

        idx_cal_rel, idx_hold_rel = next(
            divisor_3.split(
                final,
                groups=final["grupo"],
            )
        )

        candidato = tabela.copy()
        candidato["split"] = ""

        candidato.loc[
            tabela.index[idx_treino],
            "split",
        ] = "treino"

        candidato.loc[
            restante.index[idx_val_rel],
            "split",
        ] = "validacao"

        candidato.loc[
            final.index[idx_cal_rel],
            "split",
        ] = "calibracao"

        candidato.loc[
            final.index[idx_hold_rel],
            "split",
        ] = "holdout"

        distribuicao = (
            candidato.groupby(
                ["classe", "split"]
            )
            .size()
            .unstack(fill_value=0)
            .reindex(CLASSES, fill_value=0)
        )

        esperados = list(
            ALVOS_SPLIT
        )

        if any(
            split not in distribuicao.columns
            for split in esperados
        ):
            continue

        if (
            distribuicao[esperados]
            .eq(0)
            .any()
            .any()
        ):
            continue

        score = pontuar_split(candidato)

        if score < melhor_score:
            melhor = candidato
            melhor_score = score

    if melhor is None:
        raise RuntimeError(
            "Não foi possível criar um split válido."
        )

    if not treino_fixo.empty:
        fixo = treino_fixo.copy()
        fixo["split"] = "treino"

        melhor = pd.concat(
            [melhor, fixo],
            ignore_index=True,
        )

    grupos = {
        split: set(
            melhor.loc[
                melhor["split"] == split,
                "grupo",
            ]
        )
        for split in ALVOS_SPLIT
    }

    intersecoes = {}

    nomes = list(grupos)

    for indice, primeiro in enumerate(nomes):
        for segundo in nomes[
            indice + 1:
        ]:
            intersecoes[
                f"{primeiro}_{segundo}"
            ] = len(
                grupos[primeiro]
                & grupos[segundo]
            )

    if any(intersecoes.values()):
        raise RuntimeError(
            f"Vazamento entre splits: {intersecoes}"
        )

    auditoria = {
        "score": float(melhor_score),
        "intersecoes": intersecoes,
        "grupos_por_split": {
            split: len(valor)
            for split, valor in grupos.items()
        },
        "arquivos_treino_fixo": int(
            len(treino_fixo)
        ),
    }

    return melhor, auditoria


meta_split, auditoria_split = (
    criar_split_global(
        desenvolvimento
    )
)

display(
    meta_split.groupby(
        ["classe", "split"]
    )
    .agg(
        arquivos=("arquivo", "count"),
        grupos=("grupo", "nunique"),
    )
)

print(
    json.dumps(
        auditoria_split,
        ensure_ascii=False,
        indent=2,
    )
)

meta_split.to_csv(
    RESULTADOS
    / "fontes_com_split.csv",
    index=False,
)

(
    RESULTADOS
    / "auditoria_split.json"
).write_text(
    json.dumps(
        auditoria_split,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

## 7. Sequência YAMNet + MFCC

O v10 mantém a entrada temporal do v9 e adicionará pooling por
atenção no modelo.

In [ ]:
YAMNET_HANDLE = "https://tfhub.dev/google/yamnet/1"
yamnet = hub.load(YAMNET_HANDLE)
AMOSTRAS = int(CFG.sr * CFG.duracao)


def carregar_audio(
    caminho: str | Path,
) -> np.ndarray:
    audio, _ = librosa.load(
        str(caminho),
        sr=CFG.sr,
        mono=True,
    )

    audio = np.nan_to_num(
        audio.astype(np.float32)
    )

    pico = float(
        np.max(np.abs(audio))
    )

    if pico > 1.0:
        audio = audio / pico

    return audio


def carregar_audio_bytes(
    conteudo: bytes,
) -> np.ndarray:
    audio, sr = sf.read(
        io.BytesIO(conteudo),
        dtype="float32",
        always_2d=False,
    )

    if audio.ndim > 1:
        audio = audio.mean(axis=1)

    if sr != CFG.sr:
        audio = librosa.resample(
            audio,
            orig_sr=sr,
            target_sr=CFG.sr,
        )

    audio = np.nan_to_num(
        audio.astype(np.float32)
    )

    pico = float(
        np.max(np.abs(audio))
    )

    if pico > 1.0:
        audio = audio / pico

    return audio


def ajustar(
    audio: np.ndarray,
) -> np.ndarray:
    if len(audio) < AMOSTRAS:
        return np.pad(
            audio,
            (
                0,
                AMOSTRAS - len(audio),
            ),
        )

    return audio[:AMOSTRAS]


def janelas_uniformes(
    audio: np.ndarray,
    quantidade: int | None = None,
) -> list[np.ndarray]:
    quantidade = (
        quantidade
        or CFG.max_janelas_por_arquivo
    )

    if len(audio) <= AMOSTRAS:
        return [ajustar(audio)]

    inicios = np.linspace(
        0,
        len(audio) - AMOSTRAS,
        quantidade,
        dtype=int,
    )

    return [
        audio[
            inicio:
            inicio + AMOSTRAS
        ]
        for inicio in inicios
    ]


def janelas_onset(
    audio: np.ndarray,
    quantidade: int | None = None,
) -> list[np.ndarray]:
    quantidade = (
        quantidade
        or CFG.max_janelas_por_arquivo
    )

    if len(audio) <= AMOSTRAS:
        return [ajustar(audio)]

    onset = (
        librosa.onset.onset_strength(
            y=audio,
            sr=CFG.sr,
            hop_length=CFG.hop_length,
        )
    )

    picos = librosa.util.peak_pick(
        onset,
        pre_max=3,
        post_max=3,
        pre_avg=5,
        post_avg=5,
        delta=0.18,
        wait=8,
    )

    if len(picos) == 0:
        return janelas_uniformes(
            audio,
            quantidade,
        )

    ordem = np.argsort(
        onset[picos]
    )[::-1]

    janelas = []
    usados = []

    for indice in ordem:
        centro = (
            librosa.frames_to_samples(
                int(picos[indice]),
                hop_length=(
                    CFG.hop_length
                ),
            )
        )

        inicio = int(
            np.clip(
                centro
                - AMOSTRAS // 2,
                0,
                len(audio)
                - AMOSTRAS,
            )
        )

        if any(
            abs(inicio - anterior)
            < CFG.sr * 0.35
            for anterior in usados
        ):
            continue

        janelas.append(
            audio[
                inicio:
                inicio + AMOSTRAS
            ]
        )

        usados.append(inicio)

        if (
            len(janelas)
            >= quantidade
        ):
            break

    return (
        janelas
        or janelas_uniformes(
            audio,
            quantidade,
        )
    )


def extrair_features(
    audio: np.ndarray,
) -> tuple[
    np.ndarray,
    np.ndarray,
]:
    audio = ajustar(
        audio
    ).astype(np.float32)

    _, embeddings, _ = yamnet(
        tf.convert_to_tensor(
            audio,
            dtype=tf.float32,
        )
    )

    sequencia = embeddings.numpy()

    if (
        len(sequencia)
        < CFG.patches_yamnet
    ):
        sequencia = np.pad(
            sequencia,
            (
                (
                    0,
                    CFG.patches_yamnet
                    - len(sequencia),
                ),
                (0, 0),
            ),
        )

    sequencia = sequencia[
        : CFG.patches_yamnet
    ]

    mfcc = librosa.feature.mfcc(
        y=audio,
        sr=CFG.sr,
        n_mfcc=CFG.n_mfcc,
        n_fft=CFG.n_fft,
        hop_length=CFG.hop_length,
    )

    delta = librosa.feature.delta(
        mfcc,
        width=9,
    )

    delta_2 = librosa.feature.delta(
        mfcc,
        order=2,
        width=9,
    )

    rms = librosa.feature.rms(
        y=audio
    )

    centroide = (
        librosa.feature
        .spectral_centroid(
            y=audio,
            sr=CFG.sr,
        )
    )

    largura = (
        librosa.feature
        .spectral_bandwidth(
            y=audio,
            sr=CFG.sr,
        )
    )

    rolloff = (
        librosa.feature
        .spectral_rolloff(
            y=audio,
            sr=CFG.sr,
        )
    )

    zcr = (
        librosa.feature
        .zero_crossing_rate(
            audio
        )
    )

    onset = (
        librosa.onset
        .onset_strength(
            y=audio,
            sr=CFG.sr,
            hop_length=(
                CFG.hop_length
            ),
        )
    )

    mfcc_stats = np.concatenate(
        [
            mfcc.mean(axis=1),
            mfcc.std(axis=1),
            np.percentile(
                mfcc,
                [10, 50, 90],
                axis=1,
            ).reshape(-1),
            delta.mean(axis=1),
            delta.std(axis=1),
            delta_2.mean(axis=1),
            delta_2.std(axis=1),
        ]
    )

    espectral = np.asarray(
        [
            rms.mean(),
            rms.std(),
            rms.max(),
            centroide.mean(),
            centroide.std(),
            largura.mean(),
            largura.std(),
            rolloff.mean(),
            rolloff.std(),
            zcr.mean(),
            zcr.std(),
            onset.mean(),
            onset.std(),
            onset.max(),
            np.percentile(onset, 90),
            np.max(np.abs(audio)),
        ],
        dtype=np.float32,
    )

    return (
        sequencia.astype(np.float16),
        np.concatenate(
            [
                mfcc_stats,
                espectral,
            ]
        ).astype(np.float16),
    )


registros_features = []
sequencias_yamnet = []
vetores_acusticos = []


def adicionar_janelas(
    audio: np.ndarray,
    classe: str,
    grupo: str,
    fonte: str,
    arquivo: str,
    split: str = "treino",
    quantidade: int | None = None,
):
    if classe in RARAS:
        janelas = janelas_onset(
            audio,
            quantidade,
        )
    else:
        janelas = janelas_uniformes(
            audio,
            quantidade,
        )

    for janela in janelas:
        sequencia, vetor = (
            extrair_features(
                janela
            )
        )

        sequencias_yamnet.append(
            sequencia
        )

        vetores_acusticos.append(
            vetor
        )

        registros_features.append(
            {
                "classe": classe,
                "classe_id": ID[classe],
                "familia": FAMILIA[classe],
                "perigo": int(
                    classe
                    != "ambiente_normal"
                ),
                "grupo": grupo,
                "split": split,
                "fonte": fonte,
                "arquivo": arquivo,
                "suplementar": (
                    split == "treino"
                    and fonte in {
                        "FSD50K",
                        "BabyCryingKaggle",
                        "DonateACry",
                    }
                ),
            }
        )


for linha in tqdm(
    meta_split.itertuples(),
    total=len(meta_split),
    desc="Fontes-base",
):
    try:
        audio = carregar_audio(
            linha.arquivo
        )

        adicionar_janelas(
            audio=audio,
            classe=linha.classe,
            grupo=linha.grupo,
            fonte=linha.fonte,
            arquivo=linha.arquivo,
            split=linha.split,
            quantidade=(
                CFG.max_janelas_por_arquivo
            ),
        )

    except Exception as erro:
        print(
            "Falha base:",
            linha.arquivo,
            erro,
        )

print(
    "Janelas-base:",
    len(registros_features),
)

## 8. FSD50K curado

A classe `impacto` agora aceita somente sons coerentes com pancada,
batida ou golpe físico:

- `knock`
- `slam`
- `thump`
- `thud`
- `punch`

Sons acusticamente parecidos, mas de outra natureza, entram como
negativos difíceis:

- `crash`
- `crack`
- `smash`
- martelo
- furadeira
- britadeira
- fogos
- aplausos
- queda de objetos

In [ ]:
def normalizar_label_fsd(
    texto: str,
) -> str:
    return re.sub(
        r"[^a-z0-9]+",
        "_",
        str(texto).lower(),
    ).strip("_")


def mapear_labels_fsd(
    labels: list[str],
) -> str | None:
    nomes = [
        normalizar_label_fsd(
            label
        )
        for label in labels
    ]

    candidatos = set()

    if any(
        (
            "gunshot" in nome
            or "gunfire" in nome
        )
        for nome in nomes
    ):
        candidatos.add("tiro")

    if any(
        (
            "glass_breaking" in nome
            or "shatter" in nome
            or nome == "glass"
        )
        for nome in nomes
    ):
        candidatos.add(
            "vidro_quebrando"
        )

    if any(
        (
            "screaming" in nome
            or nome == "scream"
        )
        for nome in nomes
    ):
        candidatos.add(
            "grito_panico"
        )

    if any(
        (
            "crying" in nome
            or "sobbing" in nome
            or "baby_cry" in nome
            or "infant_cry" in nome
        )
        for nome in nomes
    ):
        candidatos.add(
            "choro_distress"
        )

    positivos_impacto = (
        "knock",
        "slam",
        "thump",
        "thud",
        "punch",
    )

    if any(
        any(
            termo in nome
            for termo
            in positivos_impacto
        )
        for nome in nomes
    ):
        candidatos.add("impacto")

    if len(candidatos) == 1:
        return next(iter(candidatos))

    if len(candidatos) > 1:
        return None

    negativos_dificeis = (
        "crash",
        "crack",
        "smash",
        "hammer",
        "fireworks",
        "drill",
        "jackhammer",
        "clapping",
        "applause",
        "object_drop",
        "drop",
        "door",
        "wood",
        "metal",
        "chainsaw",
        "siren",
        "engine",
        "speech",
        "laughter",
        "cough",
        "bark",
        "music",
        "footsteps",
        "vacuum",
    )

    if any(
        any(
            termo in nome
            for termo
            in negativos_dificeis
        )
        for nome in nomes
    ):
        return "ambiente_normal"

    return None


if CFG.usar_fsd50k:
    caminho_dev = Path(
        hf_hub_download(
            repo_id="Fhrozen/FSD50k",
            repo_type="dataset",
            filename="labels/dev.csv",
        )
    )

    tabela_fsd = pd.read_csv(
        caminho_dev
    )

    if "split" in tabela_fsd.columns:
        tabela_fsd = tabela_fsd[
            tabela_fsd[
                "split"
            ]
            .astype(str)
            .str.lower()
            == "train"
        ].copy()

    ids_reservados = set()

    for grupo in (
        meta_split["grupo"]
        .astype(str)
    ):
        if grupo.startswith(
            "esc50:src_"
        ):
            ids_reservados.add(
                grupo.split(
                    "esc50:src_",
                    1,
                )[1]
            )

        elif grupo.startswith(
            "urban:fsid_"
        ):
            ids_reservados.add(
                grupo.split(
                    "urban:fsid_",
                    1,
                )[1]
            )

    selecoes = {
        classe: []
        for classe in CLASSES
    }

    for linha in (
        tabela_fsd.itertuples()
    ):
        identificador = str(
            linha.fname
        )

        if (
            identificador
            in ids_reservados
        ):
            continue

        classe = mapear_labels_fsd(
            str(
                linha.labels
            ).split(",")
        )

        if classe is not None:
            selecoes[
                classe
            ].append(
                identificador
            )

    rng_fsd = random.Random(
        CFG.seed + 500
    )

    for classe, ids in (
        selecoes.items()
    ):
        ids = list(
            dict.fromkeys(ids)
        )

        rng_fsd.shuffle(ids)

        if (
            classe
            == "ambiente_normal"
        ):
            limite = (
                CFG.max_fsd_normal
            )

        elif classe == "impacto":
            limite = (
                CFG.max_fsd_impacto
            )

        else:
            limite = (
                CFG.max_fsd_por_classe
            )

        selecoes[classe] = (
            ids[:limite]
        )

    mapa_fsd = {
        identificador: classe
        for classe, ids
        in selecoes.items()
        for identificador in ids
    }

    print(
        "FSD50K selecionado:",
        {
            classe: len(ids)
            for classe, ids
            in selecoes.items()
        },
    )

    pendentes = set(
        mapa_fsd
    )

    fluxo = load_dataset(
        "philgzl/fsd50k",
        split="dev",
        streaming=True,
        features=Features(
            {
                "audio": Value(
                    "binary"
                ),
                "name": Value(
                    "string"
                ),
            }
        ),
    )

    barra = tqdm(
        total=len(pendentes),
        desc="FSD50K curado",
    )

    for item in fluxo:
        identificador = (
            Path(
                str(item["name"])
            )
            .stem
            .split("_")[0]
        )

        if (
            identificador
            not in pendentes
        ):
            continue

        classe = mapa_fsd[
            identificador
        ]

        try:
            audio = (
                carregar_audio_bytes(
                    item["audio"]
                )
            )

            adicionar_janelas(
                audio=audio,
                classe=classe,
                grupo=(
                    f"fsd50k:"
                    f"{identificador}"
                ),
                fonte="FSD50K",
                arquivo=(
                    f"stream://"
                    f"{identificador}"
                ),
                split="treino",
                quantidade=(
                    CFG
                    .max_janelas_suplementares
                ),
            )

        except Exception as erro:
            print(
                "Falha FSD50K:",
                identificador,
                erro,
            )

        pendentes.remove(
            identificador
        )

        barra.update(1)

        if not pendentes:
            break

    barra.close()

    print(
        "FSD50K não encontrados:",
        len(pendentes),
    )

else:
    print(
        "FSD50K desativado."
    )

## 9. Reforços reais de choro e fala normal

Speech Commands entra somente no treino como `ambiente_normal`.
Isso serve como negativo difícil para reduzir:

- fala normal classificada como grito;
- fala normal classificada como choro;
- falsos alertas em ambientes com pessoas conversando.

In [ ]:
# Correção do Speech Commands sem tensorflow_datasets.
# Execute em uma NOVA célula abaixo do erro.
# Não execute novamente a célula anterior, pois Baby Crying e
# Donate-a-Cry já foram adicionados às listas em memória.

if CFG.usar_speech_commands:
    print("Baixando Mini Speech Commands...")

    arquivo_zip = Path(
        tf.keras.utils.get_file(
            fname="mini_speech_commands.zip",
            origin=(
                "https://storage.googleapis.com/"
                "download.tensorflow.org/data/"
                "mini_speech_commands.zip"
            ),
            cache_dir=str(DADOS),
            cache_subdir="speech_commands",
            extract=True,
        )
    )

    candidatos_raiz = [
        arquivo_zip.parent
        / "mini_speech_commands",
        DADOS
        / "speech_commands"
        / "mini_speech_commands",
    ]

    raiz_speech = next(
        (
            caminho
            for caminho in candidatos_raiz
            if caminho.exists()
        ),
        None,
    )

    if raiz_speech is None:
        encontrados = [
            caminho
            for caminho in DADOS.rglob(
                "mini_speech_commands"
            )
            if caminho.is_dir()
        ]

        if not encontrados:
            raise FileNotFoundError(
                "A pasta mini_speech_commands "
                "não foi encontrada após a extração."
            )

        raiz_speech = encontrados[0]

    arquivos_speech = [
        caminho
        for caminho in raiz_speech.rglob("*.wav")
        if caminho.is_file()
    ]

    random.Random(
        CFG.seed + 800
    ).shuffle(arquivos_speech)

    arquivos_speech = arquivos_speech[
        : CFG.max_speech_commands
    ]

    print(
        "Speech Commands encontrados:",
        len(arquivos_speech),
    )

    adicionados_speech = 0

    for caminho in tqdm(
        arquivos_speech,
        desc="Speech Commands",
    ):
        try:
            audio = carregar_audio(
                caminho
            )

            nome = caminho.stem

            if "_nohash_" in nome:
                locutor = nome.split(
                    "_nohash_"
                )[0]
            else:
                locutor = nome

            adicionar_janelas(
                audio=audio,
                classe="ambiente_normal",
                grupo=(
                    f"speech_commands:"
                    f"{locutor}"
                ),
                fonte="SpeechCommands",
                arquivo=str(caminho),
                split="treino",
                quantidade=1,
            )

            adicionados_speech += 1

        except Exception as erro:
            print(
                "Falha Speech Commands:",
                caminho,
                erro,
            )

    print(
        "Speech Commands adicionados:",
        adicionados_speech,
    )


# Finalizar a construção dos arrays após adicionar Speech Commands.

X_yamnet = np.asarray(
    sequencias_yamnet,
    dtype=np.float16,
)

X_acustico = np.asarray(
    vetores_acusticos,
    dtype=np.float16,
)

tabela_features = pd.DataFrame(
    registros_features
)

if len(X_yamnet) != len(tabela_features):
    raise RuntimeError(
        "Quantidade de sequências YAMNet diferente "
        "da quantidade de registros."
    )

if len(X_acustico) != len(tabela_features):
    raise RuntimeError(
        "Quantidade de vetores acústicos diferente "
        "da quantidade de registros."
    )

np.save(
    CACHE / "yamnet_seq.npy",
    X_yamnet,
)

np.save(
    CACHE / "acustico.npy",
    X_acustico,
)

tabela_features.to_csv(
    CACHE / "features.csv",
    index=False,
)

print(
    "YAMNet:",
    X_yamnet.shape,
    f"{X_yamnet.nbytes / 1024**3:.2f} GB",
)

print(
    "Acústico:",
    X_acustico.shape,
    f"{X_acustico.nbytes / 1024**3:.2f} GB",
)

print(
    "\nQuantidade por classe, split e fonte:"
)

display(
    tabela_features.groupby(
        [
            "classe",
            "split",
            "fonte",
        ]
    )
    .size()
    .to_frame("janelas")
)

print(
    "\nQuantidade total por classe e split:"
)

display(
    tabela_features.groupby(
        [
            "classe",
            "split",
        ]
    )
    .size()
    .unstack(fill_value=0)
    .reindex(
        CLASSES,
        fill_value=0,
    )
)

## 10. Normalização, quatro splits e augmentation leve

O augmentation ocorre apenas em memória durante o treinamento:

- ruído gaussiano pequeno;
- mascaramento ocasional de um patch YAMNet;
- dropout em parte do vetor acústico.

Nenhum novo arquivo artificial é salvo no dataset.

In [ ]:
indices_treino = np.where(
    tabela_features["split"].to_numpy()
    == "treino"
)[0]

indices_validacao = np.where(
    tabela_features["split"].to_numpy()
    == "validacao"
)[0]

indices_calibracao = np.where(
    tabela_features["split"].to_numpy()
    == "calibracao"
)[0]

indices_holdout = np.where(
    tabela_features["split"].to_numpy()
    == "holdout"
)[0]

scaler = StandardScaler()

X_acustico_treino = scaler.fit_transform(
    X_acustico[
        indices_treino
    ].astype(np.float32)
).astype(np.float32)

X_acustico_validacao = scaler.transform(
    X_acustico[
        indices_validacao
    ].astype(np.float32)
).astype(np.float32)

X_acustico_calibracao = scaler.transform(
    X_acustico[
        indices_calibracao
    ].astype(np.float32)
).astype(np.float32)

X_acustico_holdout = scaler.transform(
    X_acustico[
        indices_holdout
    ].astype(np.float32)
).astype(np.float32)

X_yamnet_treino = X_yamnet[
    indices_treino
].astype(np.float32)

X_yamnet_validacao = X_yamnet[
    indices_validacao
].astype(np.float32)

X_yamnet_calibracao = X_yamnet[
    indices_calibracao
].astype(np.float32)

X_yamnet_holdout = X_yamnet[
    indices_holdout
].astype(np.float32)

np.savez_compressed(
    MODELOS / "normalizacao.npz",
    media=scaler.mean_.astype(np.float32),
    escala=scaler.scale_.astype(np.float32),
)

tabela_treino = (
    tabela_features.iloc[
        indices_treino
    ]
    .copy()
    .reset_index(drop=True)
)

tabela_validacao = (
    tabela_features.iloc[
        indices_validacao
    ]
    .copy()
    .reset_index(drop=True)
)

tabela_calibracao = (
    tabela_features.iloc[
        indices_calibracao
    ]
    .copy()
    .reset_index(drop=True)
)

tabela_holdout = (
    tabela_features.iloc[
        indices_holdout
    ]
    .copy()
    .reset_index(drop=True)
)


def construir_rotulos(
    tabela: pd.DataFrame,
):
    evento = (
        tabela["classe_id"]
        .astype(np.int32)
        .to_numpy()
    )

    familia = (
        tabela["familia"]
        .astype(np.int32)
        .to_numpy()
    )

    perigo = (
        tabela["perigo"]
        .astype(np.float32)
        .to_numpy()[:, None]
    )

    vocal = np.zeros(
        len(tabela),
        dtype=np.int32,
    )

    transitorio = np.zeros(
        len(tabela),
        dtype=np.int32,
    )

    impacto_normal = np.zeros(
        (len(tabela), 1),
        dtype=np.float32,
    )

    choro_normal = np.zeros(
        (len(tabela), 1),
        dtype=np.float32,
    )

    grito_normal = np.zeros(
        (len(tabela), 1),
        dtype=np.float32,
    )

    peso_vocal = np.zeros(
        len(tabela),
        dtype=np.float32,
    )

    peso_transitorio = np.zeros(
        len(tabela),
        dtype=np.float32,
    )

    peso_impacto_normal = np.zeros(
        len(tabela),
        dtype=np.float32,
    )

    peso_choro_normal = np.zeros(
        len(tabela),
        dtype=np.float32,
    )

    peso_grito_normal = np.zeros(
        len(tabela),
        dtype=np.float32,
    )

    for indice, classe in enumerate(
        tabela["classe"].tolist()
    ):
        if classe in VOCAL:
            vocal[indice] = VOCAL[classe]
            peso_vocal[indice] = 1.0

        if classe in TRANSITORIO:
            transitorio[indice] = (
                TRANSITORIO[classe]
            )
            peso_transitorio[indice] = 1.0

        if classe in {
            "ambiente_normal",
            "impacto",
        }:
            impacto_normal[
                indice,
                0,
            ] = float(
                classe == "impacto"
            )
            peso_impacto_normal[indice] = 1.0

        if classe in {
            "ambiente_normal",
            "choro_distress",
        }:
            choro_normal[
                indice,
                0,
            ] = float(
                classe
                == "choro_distress"
            )
            peso_choro_normal[indice] = 1.0

        if classe in {
            "ambiente_normal",
            "grito_panico",
        }:
            grito_normal[
                indice,
                0,
            ] = float(
                classe
                == "grito_panico"
            )
            peso_grito_normal[indice] = 1.0

    y = {
        "evento": evento,
        "familia": familia,
        "vocal": vocal,
        "transitorio": transitorio,
        "perigo": perigo,
        "impacto_normal": impacto_normal,
        "choro_normal": choro_normal,
        "grito_normal": grito_normal,
    }

    pesos = {
        "evento": np.ones(
            len(tabela),
            dtype=np.float32,
        ),
        "familia": np.ones(
            len(tabela),
            dtype=np.float32,
        ),
        "vocal": peso_vocal,
        "transitorio": peso_transitorio,
        "perigo": np.ones(
            len(tabela),
            dtype=np.float32,
        ),
        "impacto_normal": peso_impacto_normal,
        "choro_normal": peso_choro_normal,
        "grito_normal": peso_grito_normal,
    }

    return y, pesos


y_treino, pesos_treino = construir_rotulos(
    tabela_treino
)

y_validacao, pesos_validacao = construir_rotulos(
    tabela_validacao
)

y_calibracao, pesos_calibracao = construir_rotulos(
    tabela_calibracao
)

y_holdout, pesos_holdout = construir_rotulos(
    tabela_holdout
)


def entradas(
    X_yamnet_dados,
    X_acustico_dados,
):
    return {
        "yamnet_seq": X_yamnet_dados,
        "acustico": X_acustico_dados,
    }


def aumentar_lote(
    entradas_lote,
    rotulos_lote,
    pesos_lote,
):
    yamnet = tf.cast(
        entradas_lote["yamnet_seq"],
        tf.float32,
    )

    acustico = tf.cast(
        entradas_lote["acustico"],
        tf.float32,
    )

    yamnet += tf.random.normal(
        tf.shape(yamnet),
        stddev=CFG.ruido_yamnet,
    )

    acustico += tf.random.normal(
        tf.shape(acustico),
        stddev=CFG.ruido_acustico,
    )

    aplicar_mascara = (
        tf.random.uniform(())
        < CFG.prob_mascarar_patch
    )

    def mascarar_patch():
        indice_patch = tf.random.uniform(
            (),
            minval=0,
            maxval=CFG.patches_yamnet,
            dtype=tf.int32,
        )

        mascara = 1.0 - tf.one_hot(
            indice_patch,
            CFG.patches_yamnet,
            dtype=tf.float32,
        )

        mascara = mascara[
            None,
            :,
            None,
        ]

        return yamnet * mascara

    yamnet = tf.cond(
        aplicar_mascara,
        mascarar_patch,
        lambda: yamnet,
    )

    aplicar_dropout = (
        tf.random.uniform(())
        < CFG.prob_dropout_acustico
    )

    def dropout_acustico():
        mascara = tf.cast(
            tf.random.uniform(
                tf.shape(acustico)
            )
            > 0.08,
            tf.float32,
        )

        return acustico * mascara

    acustico = tf.cond(
        aplicar_dropout,
        dropout_acustico,
        lambda: acustico,
    )

    return (
        {
            "yamnet_seq": yamnet,
            "acustico": acustico,
        },
        rotulos_lote,
        pesos_lote,
    )


def dataset_finito(
    X_yamnet_dados,
    X_acustico_dados,
    y,
    pesos,
):
    return (
        tf.data.Dataset
        .from_tensor_slices(
            (
                entradas(
                    X_yamnet_dados,
                    X_acustico_dados,
                ),
                y,
                pesos,
            )
        )
        .batch(CFG.lote)
        .prefetch(1)
    )


def dataset_balanceado(
    X_yamnet_dados,
    X_acustico_dados,
    tabela,
    semente,
    indices_extras=None,
):
    datasets_classes = []
    pesos_classes = []

    for classe in CLASSES:
        indices_classe = np.where(
            tabela["classe_id"].to_numpy()
            == ID[classe]
        )[0]

        if (
            indices_extras is not None
            and classe in indices_extras
            and len(indices_extras[classe]) > 0
        ):
            indices_classe = np.concatenate(
                [
                    indices_classe,
                    indices_extras[classe],
                ]
            )

        fontes_classe = (
            tabela.iloc[
                indices_classe
            ]["fonte"]
            .astype(str)
            .to_numpy()
        )

        datasets_fontes = []

        for fonte in sorted(
            set(fontes_classe)
        ):
            mascara = (
                fontes_classe == fonte
            )

            indices_fonte = (
                indices_classe[mascara]
            )

            parte = tabela.iloc[
                indices_fonte
            ]

            y_parte, pesos_parte = construir_rotulos(
                parte
            )

            dataset_fonte = (
                tf.data.Dataset
                .from_tensor_slices(
                    (
                        entradas(
                            X_yamnet_dados[
                                indices_fonte
                            ],
                            X_acustico_dados[
                                indices_fonte
                            ],
                        ),
                        y_parte,
                        pesos_parte,
                    )
                )
                .shuffle(
                    min(
                        len(indices_fonte),
                        3000,
                    ),
                    seed=(
                        semente
                        + ID[classe]
                    ),
                    reshuffle_each_iteration=True,
                )
                .repeat()
            )

            datasets_fontes.append(
                dataset_fonte
            )

        dataset_classe = (
            tf.data.Dataset
            .sample_from_datasets(
                datasets_fontes,
                weights=[
                    1.0 / len(datasets_fontes)
                ]
                * len(datasets_fontes),
                seed=(
                    semente
                    + 100
                    + ID[classe]
                ),
                stop_on_empty_dataset=False,
            )
        )

        datasets_classes.append(
            dataset_classe
        )

        pesos_classes.append(
            PESOS_AMOSTRAGEM[classe]
        )

    return (
        tf.data.Dataset
        .sample_from_datasets(
            datasets_classes,
            weights=pesos_classes,
            seed=semente,
            stop_on_empty_dataset=False,
        )
        .batch(CFG.lote)
        .map(
            aumentar_lote,
            num_parallel_calls=(
                tf.data.AUTOTUNE
            ),
        )
        .prefetch(1)
    )


ds_treino_1 = dataset_balanceado(
    X_yamnet_treino,
    X_acustico_treino,
    tabela_treino,
    CFG.seed,
)

ds_validacao = dataset_finito(
    X_yamnet_validacao,
    X_acustico_validacao,
    y_validacao,
    pesos_validacao,
)

passos_epoca = int(
    np.clip(
        math.ceil(
            len(X_yamnet_treino)
            / CFG.lote
        ),
        450,
        1100,
    )
)

print(
    "Treino:",
    X_yamnet_treino.shape,
    X_acustico_treino.shape,
)

print(
    "Validação:",
    X_yamnet_validacao.shape,
    X_acustico_validacao.shape,
)

print(
    "Calibração:",
    X_yamnet_calibracao.shape,
    X_acustico_calibracao.shape,
)

print(
    "Holdout:",
    X_yamnet_holdout.shape,
    X_acustico_holdout.shape,
)

print(
    "Passos por época:",
    passos_epoca,
)

del X_yamnet
del X_acustico
gc.collect()

## 11. Modelo temporal com atenção e especialistas

Além da média e do máximo temporal, o v10 aprende quanto cada patch
YAMNet deve contribuir para a representação final.

In [ ]:
@tf.keras.utils.register_keras_serializable(
    package="ViolenceEdgeAI"
)
class SparseFocalLoss(
    tf.keras.losses.Loss
):
    def __init__(
        self,
        gamma=1.5,
        alpha=None,
        name="sparse_focal_loss",
    ):
        super().__init__(name=name)

        self.gamma = float(gamma)
        self.alpha = (
            None
            if alpha is None
            else [
                float(valor)
                for valor in alpha
            ]
        )

    def call(
        self,
        y_true,
        y_pred,
    ):
        y_true = tf.cast(
            tf.reshape(
                y_true,
                [-1],
            ),
            tf.int32,
        )

        y_pred = tf.clip_by_value(
            y_pred,
            1e-7,
            1.0 - 1e-7,
        )

        indices = tf.stack(
            [
                tf.range(
                    tf.shape(y_true)[0]
                ),
                y_true,
            ],
            axis=1,
        )

        p_t = tf.gather_nd(
            y_pred,
            indices,
        )

        perda = (
            -tf.pow(
                1.0 - p_t,
                self.gamma,
            )
            * tf.math.log(p_t)
        )

        if self.alpha is not None:
            alpha = tf.convert_to_tensor(
                self.alpha,
                dtype=tf.float32,
            )

            perda *= tf.gather(
                alpha,
                y_true,
            )

        return perda

    def get_config(self):
        return {
            "gamma": self.gamma,
            "alpha": self.alpha,
            "name": self.name,
        }


@tf.keras.utils.register_keras_serializable(
    package="ViolenceEdgeAI"
)
class AttentionPooling1D(
    tf.keras.layers.Layer
):
    def __init__(
        self,
        unidades=64,
        **kwargs,
    ):
        super().__init__(**kwargs)

        self.unidades = int(unidades)

        self.projecao = tf.keras.layers.Dense(
            self.unidades,
            activation="tanh",
        )

        self.score = tf.keras.layers.Dense(
            1,
            use_bias=False,
        )

    def call(self, entradas):
        logits = self.score(
            self.projecao(entradas)
        )

        pesos = tf.nn.softmax(
            logits,
            axis=1,
        )

        return tf.reduce_sum(
            entradas * pesos,
            axis=1,
        )

    def get_config(self):
        config = super().get_config()

        config.update(
            {
                "unidades": self.unidades
            }
        )

        return config


def criar_modelo():
    entrada_yamnet = tf.keras.Input(
        shape=(
            CFG.patches_yamnet,
            1024,
        ),
        dtype=tf.float32,
        name="yamnet_seq",
    )

    entrada_acustica = tf.keras.Input(
        shape=(
            X_acustico_treino.shape[1],
        ),
        dtype=tf.float32,
        name="acustico",
    )

    seq = tf.keras.layers.LayerNormalization()(
        entrada_yamnet
    )

    seq = tf.keras.layers.Conv1D(
        256,
        3,
        padding="same",
        activation="swish",
    )(seq)

    seq = tf.keras.layers.SpatialDropout1D(
        0.18
    )(seq)

    residual = tf.keras.layers.Conv1D(
        128,
        1,
        padding="same",
    )(seq)

    seq = tf.keras.layers.Conv1D(
        128,
        3,
        padding="same",
        activation="swish",
    )(seq)

    seq = tf.keras.layers.Conv1D(
        128,
        3,
        padding="same",
    )(seq)

    seq = tf.keras.layers.Add()(
        [seq, residual]
    )

    seq = tf.keras.layers.Activation(
        "swish"
    )(seq)

    seq_media = (
        tf.keras.layers
        .GlobalAveragePooling1D()
        (seq)
    )

    seq_maximo = (
        tf.keras.layers
        .GlobalMaxPooling1D()
        (seq)
    )

    seq_atencao = AttentionPooling1D(
        64,
        name="pooling_atencao",
    )(seq)

    acustico = (
        tf.keras.layers
        .BatchNormalization()
        (entrada_acustica)
    )

    acustico = tf.keras.layers.Dense(
        256,
        activation="swish",
        kernel_regularizer=(
            tf.keras.regularizers
            .l2(2e-5)
        ),
    )(acustico)

    acustico = tf.keras.layers.Dropout(
        0.25
    )(acustico)

    compartilhado = (
        tf.keras.layers
        .Concatenate()
        (
            [
                seq_media,
                seq_maximo,
                seq_atencao,
                acustico,
            ]
        )
    )

    compartilhado = tf.keras.layers.Dense(
        448,
        activation="swish",
        kernel_regularizer=(
            tf.keras.regularizers
            .l2(2e-5)
        ),
    )(compartilhado)

    compartilhado = (
        tf.keras.layers
        .BatchNormalization()
        (compartilhado)
    )

    compartilhado = tf.keras.layers.Dropout(
        0.40
    )(compartilhado)

    compartilhado = tf.keras.layers.Dense(
        224,
        activation="swish",
        kernel_regularizer=(
            tf.keras.regularizers
            .l2(1e-5)
        ),
    )(compartilhado)

    compartilhado = tf.keras.layers.Dropout(
        0.28
    )(compartilhado)

    vocal_especialista = tf.keras.layers.Dense(
        144,
        activation="swish",
        name="representacao_vocal",
    )(compartilhado)

    transitorio_especialista = (
        tf.keras.layers.Dense(
            144,
            activation="swish",
            name=(
                "representacao_transitoria"
            ),
        )
        (compartilhado)
    )

    binario_especialista = tf.keras.layers.Dense(
        112,
        activation="swish",
        name="representacao_binaria",
    )(compartilhado)

    saidas = {
        "evento": tf.keras.layers.Dense(
            len(CLASSES),
            activation="softmax",
            name="evento",
        )(compartilhado),

        "familia": tf.keras.layers.Dense(
            3,
            activation="softmax",
            name="familia",
        )(compartilhado),

        "vocal": tf.keras.layers.Dense(
            2,
            activation="softmax",
            name="vocal",
        )(vocal_especialista),

        "transitorio": tf.keras.layers.Dense(
            3,
            activation="softmax",
            name="transitorio",
        )(transitorio_especialista),

        "perigo": tf.keras.layers.Dense(
            1,
            activation="sigmoid",
            name="perigo",
        )(compartilhado),

        "impacto_normal": tf.keras.layers.Dense(
            1,
            activation="sigmoid",
            name="impacto_normal",
        )(binario_especialista),

        "choro_normal": tf.keras.layers.Dense(
            1,
            activation="sigmoid",
            name="choro_normal",
        )(binario_especialista),

        "grito_normal": tf.keras.layers.Dense(
            1,
            activation="sigmoid",
            name="grito_normal",
        )(binario_especialista),
    }

    return tf.keras.Model(
        {
            "yamnet_seq": entrada_yamnet,
            "acustico": entrada_acustica,
        },
        saidas,
        name=(
            "aed_v10_atencao_"
            "especialistas"
        ),
    )


def compilar_modelo(
    modelo,
    taxa,
):
    modelo.compile(
        optimizer=tf.keras.optimizers.AdamW(
            learning_rate=taxa,
            weight_decay=1.5e-5,
            clipnorm=1.0,
        ),
        loss={
            "evento": SparseFocalLoss(
                gamma=1.35,
                alpha=[
                    0.85,
                    1.08,
                    1.10,
                    1.42,
                    1.08,
                    1.00,
                ],
            ),
            "familia": SparseFocalLoss(
                gamma=1.15
            ),
            "vocal": SparseFocalLoss(
                gamma=1.25,
                alpha=[
                    1.08,
                    1.08,
                ],
            ),
            "transitorio": SparseFocalLoss(
                gamma=1.25,
                alpha=[
                    1.38,
                    1.05,
                    1.00,
                ],
            ),
            "perigo": "binary_crossentropy",
            "impacto_normal": "binary_crossentropy",
            "choro_normal": "binary_crossentropy",
            "grito_normal": "binary_crossentropy",
        },
        loss_weights={
            "evento": 1.00,
            "familia": 0.23,
            "vocal": 0.34,
            "transitorio": 0.42,
            "perigo": 0.26,
            "impacto_normal": 0.34,
            "choro_normal": 0.27,
            "grito_normal": 0.28,
        },
        metrics={
            "evento": [
                tf.keras.metrics
                .SparseCategoricalAccuracy(
                    name="accuracy"
                )
            ],
            "familia": [
                tf.keras.metrics
                .SparseCategoricalAccuracy(
                    name="accuracy"
                )
            ],
            "vocal": [
                tf.keras.metrics
                .SparseCategoricalAccuracy(
                    name="accuracy"
                )
            ],
            "transitorio": [
                tf.keras.metrics
                .SparseCategoricalAccuracy(
                    name="accuracy"
                )
            ],
            "perigo": [
                tf.keras.metrics.BinaryAccuracy(
                    name="accuracy"
                ),
                tf.keras.metrics.AUC(
                    name="auc"
                ),
            ],
            "impacto_normal": [
                tf.keras.metrics.BinaryAccuracy(
                    name="accuracy"
                ),
                tf.keras.metrics.AUC(
                    name="auc"
                ),
            ],
            "choro_normal": [
                tf.keras.metrics.BinaryAccuracy(
                    name="accuracy"
                ),
                tf.keras.metrics.AUC(
                    name="auc"
                ),
            ],
            "grito_normal": [
                tf.keras.metrics.BinaryAccuracy(
                    name="accuracy"
                ),
                tf.keras.metrics.AUC(
                    name="auc"
                ),
            ],
        },
        jit_compile=False,
    )


modelo = criar_modelo()

compilar_modelo(
    modelo,
    3e-4,
)

modelo.summary()

## 12. Combinação hierárquica com especialistas

O especialista `grito_normal` reduz falsos gritos e também permite
diminuir o limiar de grito sem perder tanta precisão.

In [ ]:
PESOS_INICIAIS = {
    "normal": {
        "evento": 0.42,
        "familia": 0.20,
        "perigo": 0.10,
        "impacto_normal": 0.10,
        "choro_normal": 0.09,
        "grito_normal": 0.09,
    },
    "vocal": {
        "evento": 0.41,
        "familia": 0.17,
        "ramo": 0.24,
        "perigo": 0.05,
        "choro_normal": 0.06,
        "grito_normal": 0.07,
    },
    "transitorio": {
        "evento": 0.48,
        "familia": 0.15,
        "ramo": 0.25,
        "perigo": 0.05,
        "impacto_normal": 0.07,
    },
}


def combinar_saidas(
    saidas,
    pesos,
):
    evento = np.asarray(
        saidas["evento"],
        dtype=np.float64,
    )

    familia = np.asarray(
        saidas["familia"],
        dtype=np.float64,
    )

    vocal = np.asarray(
        saidas["vocal"],
        dtype=np.float64,
    )

    transitorio = np.asarray(
        saidas["transitorio"],
        dtype=np.float64,
    )

    perigo = np.asarray(
        saidas["perigo"],
        dtype=np.float64,
    ).reshape(-1)

    impacto_normal = np.asarray(
        saidas["impacto_normal"],
        dtype=np.float64,
    ).reshape(-1)

    choro_normal = np.asarray(
        saidas["choro_normal"],
        dtype=np.float64,
    ).reshape(-1)

    grito_normal = np.asarray(
        saidas["grito_normal"],
        dtype=np.float64,
    ).reshape(-1)

    eps = 1e-8
    scores = np.zeros_like(evento)

    p = pesos["normal"]

    scores[
        :,
        ID["ambiente_normal"],
    ] = (
        np.power(
            evento[:, 0] + eps,
            p["evento"],
        )
        * np.power(
            familia[:, 0] + eps,
            p["familia"],
        )
        * np.power(
            1.0 - perigo + eps,
            p["perigo"],
        )
        * np.power(
            1.0 - impacto_normal + eps,
            p["impacto_normal"],
        )
        * np.power(
            1.0 - choro_normal + eps,
            p["choro_normal"],
        )
        * np.power(
            1.0 - grito_normal + eps,
            p["grito_normal"],
        )
    )

    p = pesos["vocal"]

    scores[
        :,
        ID["grito_panico"],
    ] = (
        np.power(
            evento[:, 1] + eps,
            p["evento"],
        )
        * np.power(
            familia[:, 1] + eps,
            p["familia"],
        )
        * np.power(
            vocal[:, 0] + eps,
            p["ramo"],
        )
        * np.power(
            perigo + eps,
            p["perigo"],
        )
        * np.power(
            grito_normal + eps,
            p["grito_normal"],
        )
    )

    scores[
        :,
        ID["choro_distress"],
    ] = (
        np.power(
            evento[:, 2] + eps,
            p["evento"],
        )
        * np.power(
            familia[:, 1] + eps,
            p["familia"],
        )
        * np.power(
            vocal[:, 1] + eps,
            p["ramo"],
        )
        * np.power(
            perigo + eps,
            p["perigo"],
        )
        * np.power(
            choro_normal + eps,
            p["choro_normal"],
        )
    )

    p = pesos["transitorio"]

    for classe, indice_ramo in (
        TRANSITORIO.items()
    ):
        indice = ID[classe]

        score = (
            np.power(
                evento[:, indice] + eps,
                p["evento"],
            )
            * np.power(
                familia[:, 2] + eps,
                p["familia"],
            )
            * np.power(
                transitorio[
                    :,
                    indice_ramo,
                ] + eps,
                p["ramo"],
            )
            * np.power(
                perigo + eps,
                p["perigo"],
            )
        )

        if classe == "impacto":
            score *= np.power(
                impacto_normal + eps,
                p["impacto_normal"],
            )
        else:
            score *= np.power(
                1.0 - impacto_normal + eps,
                p["impacto_normal"],
            )

        scores[:, indice] = score

    scores /= (
        scores.sum(
            axis=1,
            keepdims=True,
        )
        + eps
    )

    return scores


class SalvarMelhorMacroF1(
    tf.keras.callbacks.Callback
):
    def __init__(
        self,
        caminho,
        X_val,
        y_val,
        pesos,
    ):
        super().__init__()

        self.caminho = Path(caminho)
        self.X_val = X_val
        self.y_val = y_val
        self.pesos = pesos
        self.melhor = -np.inf

    def on_epoch_end(
        self,
        epoch,
        logs=None,
    ):
        logs = logs or {}

        saidas = self.model.predict(
            self.X_val,
            batch_size=192,
            verbose=0,
        )

        scores = combinar_saidas(
            saidas,
            self.pesos,
        )

        previsto = np.argmax(
            scores,
            axis=1,
        )

        macro_f1 = f1_score(
            self.y_val,
            previsto,
            average="macro",
            zero_division=0,
        )

        logs[
            "val_macro_f1_combinado"
        ] = macro_f1

        print(
            " — val_macro_f1_combinado:",
            f"{macro_f1:.4f}",
        )

        if macro_f1 > self.melhor:
            self.melhor = macro_f1

            self.model.save_weights(
                self.caminho
            )

            print(
                "Melhores pesos salvos:",
                self.caminho,
            )

## 13. Treinamento em duas etapas

A etapa 1 seleciona o melhor checkpoint exclusivamente na validação.
A etapa 2 reforça erros de ambiente normal, grito, choro e impacto.

In [ ]:
PESOS_1 = (
    MODELOS
    / "aed_v10_etapa_1.weights.h5"
)

PESOS_2 = (
    MODELOS
    / "aed_v10_etapa_2.weights.h5"
)

MODELO_FINAL = (
    MODELOS
    / "aed_v10_final.keras"
)

ENTRADAS_VALIDACAO = entradas(
    X_yamnet_validacao,
    X_acustico_validacao,
)


def callbacks_etapa(
    caminho,
    historico,
    paciencia,
):
    return [
        SalvarMelhorMacroF1(
            caminho,
            ENTRADAS_VALIDACAO,
            y_validacao["evento"],
            PESOS_INICIAIS,
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor=(
                "val_macro_f1_combinado"
            ),
            mode="max",
            patience=paciencia,
            restore_best_weights=False,
            verbose=1,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            mode="min",
            factor=0.5,
            patience=4,
            min_lr=1e-6,
            verbose=1,
        ),
        tf.keras.callbacks.CSVLogger(
            RESULTADOS / historico
        ),
    ]


modelo.fit(
    ds_treino_1,
    validation_data=ds_validacao,
    epochs=CFG.epocas_etapa_1,
    steps_per_epoch=passos_epoca,
    callbacks=callbacks_etapa(
        PESOS_1,
        "historico_etapa_1.csv",
        CFG.paciencia,
    ),
    verbose=1,
)

if not PESOS_1.exists():
    raise FileNotFoundError(
        "Pesos da etapa 1 não foram salvos."
    )

modelo_1 = criar_modelo()
modelo_1.load_weights(PESOS_1)

compilar_modelo(
    modelo_1,
    6e-5,
)

saidas_treino = modelo_1.predict(
    entradas(
        X_yamnet_treino,
        X_acustico_treino,
    ),
    batch_size=192,
    verbose=1,
)

scores_treino = combinar_saidas(
    saidas_treino,
    PESOS_INICIAIS,
)

real_treino = (
    tabela_treino[
        "classe_id"
    ].to_numpy()
)

previsto_treino = np.argmax(
    scores_treino,
    axis=1,
)

confianca_errada = scores_treino[
    np.arange(
        len(scores_treino)
    ),
    previsto_treino,
]


def selecionar_dificeis(
    classe,
    limite,
):
    indice = ID[classe]

    candidatos = np.where(
        (
            real_treino == indice
        )
        & (
            previsto_treino
            != real_treino
        )
    )[0]

    if len(candidatos) == 0:
        return np.asarray(
            [],
            dtype=np.int64,
        )

    ordem = np.argsort(
        confianca_errada[
            candidatos
        ]
    )[::-1]

    selecionados = candidatos[
        ordem[:limite]
    ]

    return np.repeat(
        selecionados,
        CFG.repeticoes_dificeis,
    )


indices_extras = {
    "ambiente_normal": selecionar_dificeis(
        "ambiente_normal",
        CFG.max_hard_normal,
    ),
    "grito_panico": selecionar_dificeis(
        "grito_panico",
        CFG.max_hard_grito,
    ),
    "choro_distress": selecionar_dificeis(
        "choro_distress",
        CFG.max_hard_choro,
    ),
    "impacto": selecionar_dificeis(
        "impacto",
        CFG.max_hard_impacto,
    ),
}

for classe, indices in (
    indices_extras.items()
):
    print(
        classe,
        "entradas difíceis extras:",
        len(indices),
    )

listas_nao_vazias = [
    valor
    for valor in indices_extras.values()
    if len(valor) > 0
]

if listas_nao_vazias:
    hard_unicos = np.unique(
        np.concatenate(
            listas_nao_vazias
        )
    )

    auditoria_hard = (
        tabela_treino.iloc[
            hard_unicos
        ]
        .copy()
    )

    auditoria_hard[
        "previsto"
    ] = [
        CLASSES[indice]
        for indice
        in previsto_treino[
            hard_unicos
        ]
    ]

    auditoria_hard[
        "confianca_errada"
    ] = confianca_errada[
        hard_unicos
    ]

    auditoria_hard.to_csv(
        RESULTADOS
        / "exemplos_dificeis.csv",
        index=False,
    )

ds_treino_2 = dataset_balanceado(
    X_yamnet_treino,
    X_acustico_treino,
    tabela_treino,
    CFG.seed + 100,
    indices_extras=indices_extras,
)

modelo_1.fit(
    ds_treino_2,
    validation_data=ds_validacao,
    epochs=CFG.epocas_etapa_2,
    steps_per_epoch=passos_epoca,
    callbacks=callbacks_etapa(
        PESOS_2,
        "historico_etapa_2.csv",
        8,
    ),
    verbose=1,
)

if not PESOS_2.exists():
    raise FileNotFoundError(
        "Pesos da etapa 2 não foram salvos."
    )

modelo_2 = criar_modelo()
modelo_2.load_weights(PESOS_2)

compilar_modelo(
    modelo_2,
    6e-5,
)

## 14. Selecionar etapa na validação e fusão na calibração

A validação escolhe entre etapa 1 e etapa 2 usando os pesos iniciais.
Depois disso, somente a calibração escolhe a combinação hierárquica.

In [ ]:
modelos = {
    "etapa_1": modelo_1,
    "etapa_2": modelo_2,
}

resultados_etapas = []

for nome, candidato in modelos.items():
    saidas = candidato.predict(
        ENTRADAS_VALIDACAO,
        batch_size=192,
        verbose=0,
    )

    scores = combinar_saidas(
        saidas,
        PESOS_INICIAIS,
    )

    previsto = np.argmax(
        scores,
        axis=1,
    )

    resultados_etapas.append(
        {
            "etapa": nome,
            "accuracy_validacao": float(
                accuracy_score(
                    y_validacao["evento"],
                    previsto,
                )
            ),
            "balanced_validacao": float(
                balanced_accuracy_score(
                    y_validacao["evento"],
                    previsto,
                )
            ),
            "f1_macro_validacao": float(
                f1_score(
                    y_validacao["evento"],
                    previsto,
                    average="macro",
                    zero_division=0,
                )
            ),
        }
    )

comparacao_etapas = (
    pd.DataFrame(
        resultados_etapas
    )
    .sort_values(
        [
            "f1_macro_validacao",
            "balanced_validacao",
            "accuracy_validacao",
        ],
        ascending=False,
    )
    .reset_index(drop=True)
)

display(comparacao_etapas)

etapa_selecionada = str(
    comparacao_etapas.loc[
        0,
        "etapa",
    ]
)

modelo_final = modelos[
    etapa_selecionada
]

ENTRADAS_CALIBRACAO = entradas(
    X_yamnet_calibracao,
    X_acustico_calibracao,
)

saidas_calibracao = modelo_final.predict(
    ENTRADAS_CALIBRACAO,
    batch_size=192,
    verbose=0,
)

CANDIDATOS_NORMAL = [
    {
        "evento": 0.48,
        "familia": 0.18,
        "perigo": 0.10,
        "impacto_normal": 0.08,
        "choro_normal": 0.08,
        "grito_normal": 0.08,
    },
    {
        "evento": 0.42,
        "familia": 0.20,
        "perigo": 0.10,
        "impacto_normal": 0.10,
        "choro_normal": 0.09,
        "grito_normal": 0.09,
    },
    {
        "evento": 0.36,
        "familia": 0.22,
        "perigo": 0.10,
        "impacto_normal": 0.11,
        "choro_normal": 0.105,
        "grito_normal": 0.105,
    },
]

CANDIDATOS_VOCAL = [
    {
        "evento": 0.48,
        "familia": 0.15,
        "ramo": 0.22,
        "perigo": 0.05,
        "choro_normal": 0.05,
        "grito_normal": 0.05,
    },
    {
        "evento": 0.41,
        "familia": 0.17,
        "ramo": 0.24,
        "perigo": 0.05,
        "choro_normal": 0.06,
        "grito_normal": 0.07,
    },
    {
        "evento": 0.35,
        "familia": 0.18,
        "ramo": 0.27,
        "perigo": 0.05,
        "choro_normal": 0.07,
        "grito_normal": 0.08,
    },
]

CANDIDATOS_TRANSITORIO = [
    {
        "evento": 0.52,
        "familia": 0.14,
        "ramo": 0.23,
        "perigo": 0.05,
        "impacto_normal": 0.06,
    },
    {
        "evento": 0.48,
        "familia": 0.15,
        "ramo": 0.25,
        "perigo": 0.05,
        "impacto_normal": 0.07,
    },
    {
        "evento": 0.41,
        "familia": 0.17,
        "ramo": 0.29,
        "perigo": 0.05,
        "impacto_normal": 0.08,
    },
]

resultados_fusao = []

for peso_normal in CANDIDATOS_NORMAL:
    for peso_vocal in CANDIDATOS_VOCAL:
        for peso_transitorio in (
            CANDIDATOS_TRANSITORIO
        ):
            pesos = {
                "normal": peso_normal,
                "vocal": peso_vocal,
                "transitorio": peso_transitorio,
            }

            scores = combinar_saidas(
                saidas_calibracao,
                pesos,
            )

            previsto = np.argmax(
                scores,
                axis=1,
            )

            resultados_fusao.append(
                {
                    "pesos": pesos,
                    "accuracy_calibracao": float(
                        accuracy_score(
                            y_calibracao["evento"],
                            previsto,
                        )
                    ),
                    "balanced_calibracao": float(
                        balanced_accuracy_score(
                            y_calibracao["evento"],
                            previsto,
                        )
                    ),
                    "f1_macro_calibracao": float(
                        f1_score(
                            y_calibracao["evento"],
                            previsto,
                            average="macro",
                            zero_division=0,
                        )
                    ),
                }
            )

comparacao_fusao = (
    pd.DataFrame(
        resultados_fusao
    )
    .sort_values(
        [
            "f1_macro_calibracao",
            "balanced_calibracao",
            "accuracy_calibracao",
        ],
        ascending=False,
    )
    .reset_index(drop=True)
)

display(
    comparacao_fusao.head(20)
)

PESOS_COMBINACAO = (
    comparacao_fusao.loc[
        0,
        "pesos",
    ]
)

modelo_final.save(
    MODELO_FINAL,
    include_optimizer=False,
)

comparacao_etapas.to_csv(
    RESULTADOS
    / "comparacao_etapas_validacao.csv",
    index=False,
)

comparacao_fusao.to_csv(
    RESULTADOS
    / "comparacao_fusao_calibracao.csv",
    index=False,
)

print(
    "Etapa selecionada:",
    etapa_selecionada,
)

print(
    "Pesos selecionados:",
    PESOS_COMBINACAO,
)

## 15. Ajustar limiares somente na calibração

In [ ]:
scores_calibracao = combinar_saidas(
    saidas_calibracao,
    PESOS_COMBINACAO,
)

real_calibracao = y_calibracao[
    "evento"
]

limiares_evento = {
    "ambiente_normal": 0.0
}

linhas_limiares = []

for indice, classe in enumerate(
    CLASSES
):
    if classe == "ambiente_normal":
        continue

    real_binario = (
        real_calibracao == indice
    ).astype(int)

    candidatos = []

    for limiar in np.arange(
        0.12,
        0.81,
        0.01,
    ):
        previsto_binario = (
            scores_calibracao[
                :,
                indice,
            ]
            >= limiar
        ).astype(int)

        candidatos.append(
            {
                "limiar": float(limiar),
                "precision": float(
                    precision_score(
                        real_binario,
                        previsto_binario,
                        zero_division=0,
                    )
                ),
                "recall": float(
                    recall_score(
                        real_binario,
                        previsto_binario,
                        zero_division=0,
                    )
                ),
                "f1": float(
                    f1_score(
                        real_binario,
                        previsto_binario,
                        zero_division=0,
                    )
                ),
            }
        )

    if classe == "grito_panico":
        precisao_minima = 0.78
        beta_recall = 1.25

    elif classe == "impacto":
        precisao_minima = 0.68
        beta_recall = 1.20

    elif classe == "choro_distress":
        precisao_minima = 0.76
        beta_recall = 1.00

    else:
        precisao_minima = 0.74
        beta_recall = 1.00

    aceitaveis = [
        item
        for item in candidatos
        if (
            item["precision"]
            >= precisao_minima
        )
    ]

    def pontuacao(item):
        beta_2 = beta_recall ** 2

        denominador = (
            beta_2
            * item["precision"]
            + item["recall"]
        )

        f_beta = (
            0.0
            if denominador == 0
            else (
                (1 + beta_2)
                * item["precision"]
                * item["recall"]
                / denominador
            )
        )

        return (
            f_beta,
            item["f1"],
            item["recall"],
        )

    melhor = max(
        aceitaveis or candidatos,
        key=pontuacao,
    )

    limiares_evento[classe] = round(
        melhor["limiar"],
        2,
    )

    linhas_limiares.append(
        {
            "classe": classe,
            "precisao_minima": (
                precisao_minima
            ),
            "beta_recall": beta_recall,
            **melhor,
        }
    )

tabela_limiares = pd.DataFrame(
    linhas_limiares
)

display(tabela_limiares)

(
    MODELOS / "limiares.json"
).write_text(
    json.dumps(
        {
            "limiares_evento": (
                limiares_evento
            ),
            "minimo_consecutivas": {
                "grito_panico": 2,
                "choro_distress": 3,
                "impacto": 2,
                "vidro_quebrando": 2,
                "tiro": 2,
            },
            "pesos_combinacao": (
                PESOS_COMBINACAO
            ),
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)


def aplicar_limiares(
    scores,
):
    resultado = []

    for vetor in scores:
        indice = int(
            np.argmax(vetor)
        )

        classe = CLASSES[indice]

        if classe == "ambiente_normal":
            resultado.append(indice)

        elif (
            vetor[indice]
            >= limiares_evento[classe]
        ):
            resultado.append(indice)

        else:
            resultado.append(
                ID["ambiente_normal"]
            )

    return np.asarray(
        resultado,
        dtype=np.int32,
    )

## 16. Avaliação oficial no holdout bloqueado

Esta é a primeira etapa que utiliza o holdout.

In [ ]:
ENTRADAS_HOLDOUT = entradas(
    X_yamnet_holdout,
    X_acustico_holdout,
)

saidas_holdout = modelo_final.predict(
    ENTRADAS_HOLDOUT,
    batch_size=192,
    verbose=1,
)

scores_holdout = combinar_saidas(
    saidas_holdout,
    PESOS_COMBINACAO,
)

pred_evento = aplicar_limiares(
    scores_holdout
)

real_evento = y_holdout[
    "evento"
]

accuracy = accuracy_score(
    real_evento,
    pred_evento,
)

balanced = balanced_accuracy_score(
    real_evento,
    pred_evento,
)

f1_macro = f1_score(
    real_evento,
    pred_evento,
    average="macro",
    zero_division=0,
)

relatorio = pd.DataFrame(
    classification_report(
        real_evento,
        pred_evento,
        labels=np.arange(
            len(CLASSES)
        ),
        target_names=CLASSES,
        output_dict=True,
        digits=4,
        zero_division=0,
    )
).T

display(relatorio)

f1_minimo = float(
    relatorio.loc[
        CLASSES,
        "f1-score",
    ].min()
)

atingiu_meta = bool(
    accuracy >= CFG.meta_accuracy
    and f1_macro >= CFG.meta_f1_macro
    and f1_minimo >= CFG.meta_f1_classe
)

resumo = {
    "accuracy_holdout": float(
        accuracy
    ),
    "balanced_accuracy_holdout": float(
        balanced
    ),
    "f1_macro_holdout": float(
        f1_macro
    ),
    "menor_f1_classe_holdout": float(
        f1_minimo
    ),
    "atingiu_meta": atingiu_meta,
    "etapa_selecionada": (
        etapa_selecionada
    ),
    "pesos_combinacao": (
        PESOS_COMBINACAO
    ),
    "limiares_evento": (
        limiares_evento
    ),
    "auditoria_split": (
        auditoria_split
    ),
    "metas": {
        "accuracy": CFG.meta_accuracy,
        "f1_macro": CFG.meta_f1_macro,
        "f1_classe": CFG.meta_f1_classe,
    },
}

print(
    json.dumps(
        resumo,
        ensure_ascii=False,
        indent=2,
    )
)

relatorio.to_csv(
    RESULTADOS
    / "metricas_holdout_por_classe.csv"
)

(
    RESULTADOS
    / "resumo_metricas_holdout.json"
).write_text(
    json.dumps(
        resumo,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

matriz = confusion_matrix(
    real_evento,
    pred_evento,
    labels=np.arange(
        len(CLASSES)
    ),
)

plt.figure(figsize=(10, 8))
plt.imshow(
    matriz,
    cmap="Blues",
)
plt.colorbar()

plt.title(
    "Matriz de confusão — AED v10 holdout"
)

plt.xticks(
    range(len(CLASSES)),
    CLASSES,
    rotation=45,
    ha="right",
)

plt.yticks(
    range(len(CLASSES)),
    CLASSES,
)

for linha in range(
    len(CLASSES)
):
    for coluna in range(
        len(CLASSES)
    ):
        plt.text(
            coluna,
            linha,
            str(
                matriz[
                    linha,
                    coluna,
                ]
            ),
            ha="center",
            va="center",
        )

plt.xlabel("Previsto")
plt.ylabel("Real")
plt.tight_layout()

plt.savefig(
    RESULTADOS
    / "matriz_confusao_holdout.png",
    dpi=180,
)

plt.show()

## 17. Avaliação por gravação no holdout

- normal: mediana;
- grito: percentil 75;
- choro: média das três maiores janelas, com persistência maior;
- transitórios: média das duas maiores janelas.

In [ ]:
tabela_holdout_resultado = (
    tabela_holdout.copy()
)

for indice, classe in enumerate(
    CLASSES
):
    tabela_holdout_resultado[
        f"prob_{classe}"
    ] = scores_holdout[
        :,
        indice,
    ]

linhas = []

for grupo, parte in (
    tabela_holdout_resultado
    .groupby("grupo")
):
    vetor = np.zeros(
        len(CLASSES),
        dtype=np.float64,
    )

    for indice, classe in enumerate(
        CLASSES
    ):
        valores = (
            parte[
                f"prob_{classe}"
            ]
            .to_numpy()
        )

        if classe == "ambiente_normal":
            vetor[indice] = float(
                np.median(valores)
            )

        elif classe == "grito_panico":
            vetor[indice] = float(
                np.percentile(
                    valores,
                    75,
                )
            )

        elif classe == "choro_distress":
            ordenados = np.sort(
                valores
            )[::-1]

            k = min(
                3,
                len(ordenados),
            )

            vetor[indice] = float(
                ordenados[:k].mean()
            )

        else:
            ordenados = np.sort(
                valores
            )[::-1]

            k = min(
                2,
                len(ordenados),
            )

            vetor[indice] = float(
                ordenados[:k].mean()
            )

    vetor /= (
        vetor.sum() + 1e-8
    )

    previsto = int(
        aplicar_limiares(
            vetor[None, :]
        )[0]
    )

    real = int(
        parte[
            "classe_id"
        ]
        .mode()
        .iloc[0]
    )

    linhas.append(
        {
            "grupo": grupo,
            "real": real,
            "previsto": previsto,
            "real_nome": CLASSES[real],
            "previsto_nome": (
                CLASSES[previsto]
            ),
            "confianca": float(
                vetor[previsto]
            ),
        }
    )

gravacoes = pd.DataFrame(
    linhas
)

metricas_gravacao = {
    "accuracy": float(
        accuracy_score(
            gravacoes["real"],
            gravacoes["previsto"],
        )
    ),
    "balanced_accuracy": float(
        balanced_accuracy_score(
            gravacoes["real"],
            gravacoes["previsto"],
        )
    ),
    "f1_macro": float(
        f1_score(
            gravacoes["real"],
            gravacoes["previsto"],
            average="macro",
            zero_division=0,
        )
    ),
}

print(
    json.dumps(
        metricas_gravacao,
        indent=2,
    )
)

gravacoes.to_csv(
    RESULTADOS
    / "predicoes_holdout_por_gravacao.csv",
    index=False,
)

## 18. Avaliação externa AOB

O AOB de avaliação não participa do treino, validação,
calibração nem holdout.

In [ ]:
metricas_externas = None

if externo.empty:
    print(
        "Conjunto externo AOB ausente."
    )

else:
    seq_externo = []
    acustico_externo = []
    registros_externos = []

    for linha in tqdm(
        externo.itertuples(),
        total=len(externo),
        desc="AOB externo",
    ):
        try:
            audio = carregar_audio(
                linha.arquivo
            )

            if linha.classe in RARAS:
                janelas = janelas_onset(
                    audio,
                    CFG.max_janelas_por_arquivo,
                )
            else:
                janelas = janelas_uniformes(
                    audio,
                    CFG.max_janelas_por_arquivo,
                )

            for janela in janelas:
                sequencia, vetor = (
                    extrair_features(
                        janela
                    )
                )

                seq_externo.append(
                    sequencia
                )

                acustico_externo.append(
                    vetor
                )

                registros_externos.append(
                    {
                        "classe": linha.classe,
                        "classe_id": (
                            linha.classe_id
                        ),
                        "grupo": linha.grupo,
                    }
                )

        except Exception as erro:
            print(
                "Falha externa:",
                linha.arquivo,
                erro,
            )

    if seq_externo:
        X_seq_externo = np.asarray(
            seq_externo,
            dtype=np.float32,
        )

        X_acustico_externo = (
            scaler.transform(
                np.asarray(
                    acustico_externo,
                    dtype=np.float32,
                )
            )
            .astype(np.float32)
        )

        tabela_externa = pd.DataFrame(
            registros_externos
        )

        saidas_externas = modelo_final.predict(
            entradas(
                X_seq_externo,
                X_acustico_externo,
            ),
            batch_size=192,
            verbose=1,
        )

        scores_externos = combinar_saidas(
            saidas_externas,
            PESOS_COMBINACAO,
        )

        previstos_externos = aplicar_limiares(
            scores_externos
        )

        reais_externos = (
            tabela_externa[
                "classe_id"
            ].to_numpy()
        )

        presentes = sorted(
            np.unique(
                reais_externos
            ).tolist()
        )

        metricas_externas = {
            "accuracy": float(
                accuracy_score(
                    reais_externos,
                    previstos_externos,
                )
            ),
            "balanced_accuracy": float(
                balanced_accuracy_score(
                    reais_externos,
                    previstos_externos,
                )
            ),
            "f1_macro_classes_presentes": float(
                f1_score(
                    reais_externos,
                    previstos_externos,
                    labels=presentes,
                    average="macro",
                    zero_division=0,
                )
            ),
            "classes_presentes": [
                CLASSES[indice]
                for indice in presentes
            ],
        }

        print(
            json.dumps(
                metricas_externas,
                ensure_ascii=False,
                indent=2,
            )
        )

        (
            RESULTADOS
            / "metricas_externas_aob.json"
        ).write_text(
            json.dumps(
                metricas_externas,
                ensure_ascii=False,
                indent=2,
            ),
            encoding="utf-8",
        )

## 19. Exportação TFLite e paridade

In [ ]:
saidas_modelo = modelo_final(
    {
        "yamnet_seq": (
            modelo_final.get_layer(
                "yamnet_seq"
            ).output
        ),
        "acustico": (
            modelo_final.get_layer(
                "acustico"
            ).output
        ),
    }
)

ORDEM_SAIDAS = [
    "evento",
    "familia",
    "vocal",
    "transitorio",
    "perigo",
    "impacto_normal",
    "choro_normal",
    "grito_normal",
]

saida_unificada = (
    tf.keras.layers
    .Concatenate(
        name="saida_unificada"
    )
    (
        [
            saidas_modelo[nome]
            for nome in ORDEM_SAIDAS
        ]
    )
)

modelo_exportacao = tf.keras.Model(
    modelo_final.inputs,
    saida_unificada,
    name="aed_v10_exportacao",
)

TFLITE_FLOAT32 = (
    MODELOS
    / "aed_v10_classificador_float32.tflite"
)

TFLITE_FLOAT16 = (
    MODELOS
    / "aed_v10_classificador_float16.tflite"
)

conversor = (
    tf.lite.TFLiteConverter
    .from_keras_model(
        modelo_exportacao
    )
)

TFLITE_FLOAT32.write_bytes(
    conversor.convert()
)

conversor = (
    tf.lite.TFLiteConverter
    .from_keras_model(
        modelo_exportacao
    )
)

conversor.optimizations = [
    tf.lite.Optimize.DEFAULT
]

conversor.target_spec.supported_types = [
    tf.float16
]

TFLITE_FLOAT16.write_bytes(
    conversor.convert()
)


def prever_tflite(
    caminho,
    seq,
    acustico,
):
    interpretador = tf.lite.Interpreter(
        model_path=str(caminho)
    )

    interpretador.allocate_tensors()

    detalhes_entrada = (
        interpretador.get_input_details()
    )

    detalhe_saida = (
        interpretador.get_output_details()[0]
    )

    resultados = []

    for indice in range(len(seq)):
        for detalhe in detalhes_entrada:
            formato = tuple(
                detalhe["shape"]
            )

            if len(formato) == 3:
                valor = seq[
                    indice:
                    indice + 1
                ]
            else:
                valor = acustico[
                    indice:
                    indice + 1
                ]

            interpretador.set_tensor(
                detalhe["index"],
                valor.astype(
                    detalhe["dtype"]
                ),
            )

        interpretador.invoke()

        resultados.append(
            interpretador.get_tensor(
                detalhe_saida["index"]
            )[0]
        )

    return np.asarray(resultados)


rng = np.random.default_rng(
    CFG.seed
)

quantidade = min(
    250,
    len(X_yamnet_holdout),
)

indices = rng.choice(
    len(X_yamnet_holdout),
    size=quantidade,
    replace=False,
)

keras_saida = modelo_exportacao.predict(
    entradas(
        X_yamnet_holdout[indices],
        X_acustico_holdout[indices],
    ),
    batch_size=64,
    verbose=0,
)

linhas_paridade = []

for caminho in (
    TFLITE_FLOAT32,
    TFLITE_FLOAT16,
):
    tflite_saida = prever_tflite(
        caminho,
        X_yamnet_holdout[indices],
        X_acustico_holdout[indices],
    )

    linhas_paridade.append(
        {
            "modelo": caminho.name,
            "concordancia_evento": float(
                np.mean(
                    np.argmax(
                        tflite_saida[:, :6],
                        axis=1,
                    )
                    == np.argmax(
                        keras_saida[:, :6],
                        axis=1,
                    )
                )
            ),
            "diferenca_maxima": float(
                np.max(
                    np.abs(
                        tflite_saida
                        - keras_saida
                    )
                )
            ),
        }
    )

paridade = pd.DataFrame(
    linhas_paridade
)

display(paridade)

paridade.to_csv(
    RESULTADOS
    / "paridade_tflite.csv",
    index=False,
)

if (
    paridade[
        "concordancia_evento"
    ].min()
    < CFG.meta_paridade
):
    raise RuntimeError(
        "Paridade TFLite abaixo da meta."
    )

## 20. Teste com áudio externo opcional

In [ ]:
enviados = files.upload()

arquivos_audio = [
    nome
    for nome in enviados
    if (
        Path(nome)
        .suffix
        .lower()
        in EXTENSOES
    )
]

if len(arquivos_audio) != 1:
    raise RuntimeError(
        "Envie exatamente um áudio."
    )

AUDIO_EXTERNO = Path(
    arquivos_audio[0]
).resolve()

display(
    Audio(
        str(AUDIO_EXTERNO)
    )
)

audio = carregar_audio(
    AUDIO_EXTERNO
)

if len(audio) <= AMOSTRAS:
    inicios = [0]

else:
    passo = int(
        CFG.sr * CFG.passo
    )

    inicios = list(
        range(
            0,
            len(audio)
            - AMOSTRAS
            + 1,
            passo,
        )
    )

seq_audio = []
acustico_audio = []

for inicio in inicios:
    sequencia, vetor = (
        extrair_features(
            audio[
                inicio:
                inicio + AMOSTRAS
            ]
        )
    )

    seq_audio.append(
        sequencia
    )

    acustico_audio.append(
        vetor
    )

seq_audio = np.asarray(
    seq_audio,
    dtype=np.float32,
)

acustico_audio = (
    scaler.transform(
        np.asarray(
            acustico_audio,
            dtype=np.float32,
        )
    )
    .astype(np.float32)
)

saidas_audio = (
    modelo_final.predict(
        entradas(
            seq_audio,
            acustico_audio,
        ),
        batch_size=64,
        verbose=0,
    )
)

scores_audio = combinar_saidas(
    saidas_audio,
    PESOS_COMBINACAO,
)

previsto_audio = (
    aplicar_limiares(
        scores_audio
    )
)

linhas = []

for inicio, indice, vetor in zip(
    inicios,
    previsto_audio,
    scores_audio,
    strict=True,
):
    linhas.append(
        {
            "inicio": (
                inicio / CFG.sr
            ),
            "fim": (
                inicio / CFG.sr
                + CFG.duracao
            ),
            "classe": (
                CLASSES[indice]
            ),
            "confianca": float(
                vetor[indice]
            ),
            "ativo": bool(
                indice
                != ID[
                    "ambiente_normal"
                ]
            ),
        }
    )

timeline = pd.DataFrame(
    linhas
)

consecutivas = 0
anterior = None
alertas = []

for linha in (
    timeline.itertuples()
):
    if (
        linha.ativo
        and linha.classe
        == anterior
    ):
        consecutivas += 1

    elif linha.ativo:
        consecutivas = 1

    else:
        consecutivas = 0

    anterior = (
        linha.classe
        if linha.ativo
        else None
    )

    alertas.append(
        consecutivas >= 2
    )

timeline[
    "alerta_temporal"
] = alertas

display(
    timeline.sort_values(
        [
            "alerta_temporal",
            "confianca",
        ],
        ascending=False,
    ).head(50)
)

timeline.to_csv(
    RESULTADOS
    / "timeline_audio_externo.csv",
    index=False,
)

## 21. Pacote final

In [ ]:
configuracao_final = {
    "versao": "aed_v10",
    "classes": CLASSES,
    "configuracao": asdict(CFG),
    "yamnet_handle": YAMNET_HANDLE,
    "entrada_yamnet": [
        CFG.patches_yamnet,
        1024,
    ],
    "dimensao_acustica": int(
        X_acustico_treino.shape[1]
    ),
    "ordem_saidas_tflite": (
        ORDEM_SAIDAS
    ),
    "etapa_selecionada": (
        etapa_selecionada
    ),
    "pesos_combinacao": (
        PESOS_COMBINACAO
    ),
    "limiares_evento": (
        limiares_evento
    ),
    "metricas_holdout": resumo,
    "metricas_gravacao_holdout": (
        metricas_gravacao
    ),
    "metricas_externas_aob": (
        metricas_externas
    ),
    "auditoria_split": (
        auditoria_split
    ),
    "atingiu_meta": (
        atingiu_meta
    ),
}

(
    MODELOS
    / "configuracao_aed_v10.json"
).write_text(
    json.dumps(
        configuracao_final,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

texto_implantacao = (
    '''
    AED v10

    1. Capturar áudio mono em 16 kHz.
    2. Usar janelas de {duracao:.1f} segundos.
    3. Executar YAMNet e conservar {patches} embeddings.
    4. Calcular MFCC, delta, delta-delta e descritores.
    5. Normalizar a entrada acústica com normalizacao.npz.
    6. Executar aed_v10_classificador_float16.tflite.
    7. Separar a saída na ordem de configuracao_aed_v10.json.
    8. Aplicar os pesos de combinação e os limiares.
    9. Para choro, exigir três janelas consecutivas.
    10. Para os demais perigos, exigir duas janelas consecutivas.
    11. O VAD continua separado e aciona somente Whisper/NLP.
    '''
    .format(
        duracao=CFG.duracao,
        patches=CFG.patches_yamnet,
    )
    .strip()
)

(
    MODELOS
    / "README_IMPLANTACAO.txt"
).write_text(
    texto_implantacao,
    encoding="utf-8",
)

if ENTREGA.exists():
    shutil.rmtree(ENTREGA)

ENTREGA.mkdir(
    parents=True,
    exist_ok=True,
)

for arquivo in MODELOS.glob("*"):
    if arquivo.is_file():
        shutil.copy2(
            arquivo,
            ENTREGA / arquivo.name,
        )

for arquivo in RESULTADOS.glob("*"):
    if arquivo.is_file():
        shutil.copy2(
            arquivo,
            ENTREGA / arquivo.name,
        )

ZIP_FINAL = shutil.make_archive(
    "/content/aed_v10_entrega",
    "zip",
    root_dir=str(ENTREGA),
)

print(
    "Meta atingida:",
    atingiu_meta,
)

print(
    "Modelo:",
    MODELO_FINAL,
)

print(
    "ZIP:",
    ZIP_FINAL,
)

files.download(ZIP_FINAL)

## Observações

- O holdout do v10 é novo e não deve ser usado para ajustar o modelo.
- Os resultados do v10 não são diretamente comparáveis ao teste do v9,
  porque a divisão final foi refeita para criar calibração e holdout.
- Speech Commands serve apenas como fala normal negativa.
- TUT Rare e AOB desenvolvimento ficam somente no treino.
- AOB avaliação permanece externo.
- Não apague o v9 até o v10 comprovar ganho consistente.
- Caso impacto continue abaixo de 80%, o próximo passo real é ampliar
  gravações de pancadas e golpes, não aumentar épocas.